In [ ]:
import numpy as np
import pandas as pd
import io
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib import animation, rc
rc("animation", html="jshtml")
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D

from IPython.display import display, HTML, clear_output, FileLink
from ipywidgets import (
    VBox, HBox, Layout, Dropdown, FloatSlider, IntSlider, Checkbox,
    FileUpload, Label, Tab, Output, Button, Accordion, HTML as WHTML
)

from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.pipeline import make_pipeline

from sklearn.metrics import roc_curve, auc, mean_squared_error, r2_score, confusion_matrix
from sklearn.base import BaseEstimator, ClassifierMixin

from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, RandomForestRegressor, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import Perceptron, LogisticRegression, LinearRegression, Lasso, ElasticNet, Ridge, SGDClassifier
from sklearn.svm import SVR

import datetime
import warnings
warnings.filterwarnings("ignore")

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

data_cache = {}

def compute_roc_auc(clf, X_test, y_test):
    scores = None
    if hasattr(clf, "decision_function"):
        try:
            scores = clf.decision_function(X_test)
        except Exception:
            scores = None
    if scores is None and hasattr(clf, "predict_proba"):
        try:
            scores = clf.predict_proba(X_test)[:, 1]
        except Exception:
            scores = None
    if scores is None:
        try:
            scores = clf.predict(X_test)
        except Exception:
            scores = np.zeros(len(X_test))

    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_auc = auc(fpr, tpr)
    return fpr, tpr, roc_auc

def draw_summary_box(ax, lines):
    text = "\n".join(lines)
    ax.set_axis_off()
    ax.text(
        0.01,
        0.5,
        text,
        transform=ax.transAxes,
        va="center",
        ha="left",
        fontsize=9,
        family="monospace",
        bbox=dict(
            boxstyle="round",
            facecolor="#222222",
            edgecolor="#aaaaaa",
            alpha=0.9
        ),
    )

def plot_learning_curve_ax(ax, estimator, X, y, task_type, random_state):
    try:
        scoring = "accuracy" if task_type == "cls" else "r2"
        train_sizes = np.linspace(0.2, 1.0, 5)

        train_sizes_abs, train_scores, val_scores = learning_curve(
            estimator,
            X,
            y,
            train_sizes=train_sizes,
            cv=3,
            scoring=scoring,
            shuffle=True,
            random_state=random_state,
        )

        train_mean = train_scores.mean(axis=1)
        val_mean = val_scores.mean(axis=1)

        ax.plot(train_sizes_abs, train_mean, "o-", label="Train")
        ax.plot(train_sizes_abs, val_mean, "o-", label="Validation")
        ax.set_xlabel("Training examples")
        ax.set_ylabel(scoring)
        ax.set_title("Learning curve", fontsize=10)
        ax.grid(alpha=0.3)
        ax.legend()
    except Exception as e:
        ax.text(
            0.5,
            0.5,
            f"LC error:\n{e}",
            ha="center",
            va="center",
            fontsize=8,
        )
        ax.set_axis_off()

def get_model_family(model):
    if isinstance(model, (LogisticRegression, LinearSVC, Perceptron, LinearDiscriminantAnalysis)):
        return "linear"

    if isinstance(model, (RandomForestClassifier, AdaBoostClassifier, DecisionTreeClassifier)):
        return "tree_ensemble"

    if isinstance(model, KNeighborsClassifier):
        return "knn"

    if isinstance(model, SVC):
        return "svm"

    if isinstance(model, MLPClassifier):
        return "mlp"

    return "other"

def get_reg_model_family(model):
    if isinstance(model, LinearRegression):
        return "linear"

    if isinstance(model, (RandomForestRegressor, DecisionTreeRegressor)):
        return "tree_ensemble"

    if isinstance(model, KNeighborsRegressor):
        return "knn"

    if isinstance(model, MLPRegressor):
        return "mlp"

    return "other"

def make_linear_blobs(n_samples=300, noise=0.1, random_state=42):
    X, y = make_blobs(
        n_samples=n_samples,
        centers=[(-2, -2), (2, 2)],
        cluster_std=noise + 0.3,
        random_state=random_state
    )
    return X, y

def make_linear_shifted(n_samples=300, noise=0.1, random_state=42):
    rng = np.random.RandomState(random_state)
    X1 = rng.randn(n_samples // 2, 2) * (0.3 + noise) + np.array([-2, 0])
    X2 = rng.randn(n_samples // 2, 2) * (0.3 + noise) + np.array([2, 0])
    X = np.vstack([X1, X2])
    y = np.hstack([np.zeros(len(X1)), np.ones(len(X2))])
    return X, y

def make_linear_vertical(n_samples=300, noise=0.1, random_state=42):
    rng = np.random.RandomState(random_state)
    X1 = rng.randn(n_samples // 2, 2) * (0.3 + noise) + np.array([0, -2])
    X2 = rng.randn(n_samples // 2, 2) * (0.3 + noise) + np.array([0, 2])
    X = np.vstack([X1, X2])
    y = np.hstack([np.zeros(len(X1)), np.ones(len(X2))])
    return X, y

def make_spiral(n_samples=300, noise=0.2, random_state=None):
    rng = np.random.RandomState(random_state)
    n = n_samples // 2
    theta = np.sqrt(rng.rand(n)) * 2 * np.pi
    r_a = 2 * theta + np.pi
    data_a = np.c_[r_a * np.cos(theta), r_a * np.sin(theta)]
    theta = np.sqrt(rng.rand(n)) * 2 * np.pi
    r_b = -2 * theta - np.pi
    data_b = np.c_[r_b * np.cos(theta), r_b * np.sin(theta)]
    X = np.vstack([data_a, data_b])
    X += rng.normal(scale=noise, size=X.shape)
    y = np.hstack([np.zeros(n, dtype=int), np.ones(n, dtype=int)])
    return X, y

def make_xor(n_samples=300, noise=0.2, random_state=None):
    rng = np.random.RandomState(random_state)
    X = rng.randn(n_samples, 2)
    y = (X[:, 0] * X[:, 1] > 0).astype(int)
    X += rng.normal(scale=noise, size=X.shape)
    return X, y

def make_checkerboard(n_samples=300, noise=0.1, random_state=None):
    rng = np.random.RandomState(random_state)
    X = rng.uniform(-2, 2, size=(n_samples, 2))
    y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(int)
    X += rng.normal(scale=noise * 0.2, size=X.shape)
    return X, y

def generate_linear_data(kind, n_samples, noise_std, random_state):
    if kind == "blobs":
        return make_linear_blobs(n_samples, noise_std, random_state)
    if kind == "shifted":
        return make_linear_shifted(n_samples, noise_std, random_state)
    if kind == "vertical":
        return make_linear_vertical(n_samples, noise_std, random_state)
    return make_linear_blobs(n_samples, noise_std, random_state)

def generate_nonlinear_data(kind, n_samples, noise_std, random_state):
    if kind == "moons":
        return make_moons(n_samples=n_samples, noise=noise_std, random_state=random_state)
    if kind == "circles":
        return make_circles(n_samples=n_samples, noise=noise_std, factor=0.5, random_state=random_state)
    if kind == "blobs":
        X, y = make_blobs(n_samples=n_samples, centers=[(-1, -1), (1, 1)],
                          cluster_std=0.8, random_state=random_state)
        return X, y
    if kind == "xor":
        return make_xor(n_samples, noise_std, random_state)
    if kind == "spiral":
        return make_spiral(n_samples, noise_std, random_state)
    if kind == "checkerboard":
        return make_checkerboard(n_samples, noise_std, random_state)
    return make_moons(n_samples=n_samples, noise=noise_std, random_state=random_state)

def generate_regression_data(kind, n_samples, noise_std, random_state):
    rng = np.random.RandomState(random_state)
    X = rng.uniform(-3, 3, size=(n_samples, 2))
    if kind == "linear":
        w = np.array([2.0, -3.0])
        y = X @ w + 10
    elif kind == "quadratic":
        y = 2 * X[:, 0] ** 2 - 3 * X[:, 1] + 5
    else:
        y = X[:, 0] * X[:, 1] + 0.5 * X[:, 0] ** 2
    y = y + rng.normal(scale=noise_std, size=y.shape)
    return X, y

def apply_noise_outliers_classification(X, y, noise_pct, outliers_pct, random_state):
    rng = np.random.RandomState(random_state + 123)
    X = X.copy()
    y = y.copy()
    n = len(y)

    num_noise = int(noise_pct / 100.0 * n)
    if num_noise > 0:
        idx = rng.choice(n, size=num_noise, replace=False)
        y[idx] = 1 - y[idx]

    num_out = int(outliers_pct / 100.0 * n)
    if num_out > 0:
        idx = rng.choice(n, size=num_out, replace=False)
        X[idx] = rng.uniform(-6, 6, size=(num_out, X.shape[1]))
        y[idx] = rng.randint(0, 2, size=num_out)

    return X, y

def apply_noise_outliers_regression(X, y, noise_pct, outliers_pct, random_state):
    rng = np.random.RandomState(random_state + 456)
    X = X.copy()
    y = y.copy()
    n = len(y)

    num_noise = int(noise_pct / 100.0 * n)
    if num_noise > 0:
        idx = rng.choice(n, size=num_noise, replace=False)
        y_std = y.std() if y.std() > 0 else 1.0
        y[idx] += rng.normal(scale=0.5 * y_std, size=num_noise)

    num_out = int(outliers_pct / 100.0 * n)
    if num_out > 0:
        idx = rng.choice(n, size=num_out, replace=False)
        y_std = y.std() if y.std() > 0 else 1.0
        y[idx] += rng.normal(loc=5 * y_std, scale=2 * y_std, size=num_out)

    return X, y

linear_model_names = [
    "Perceptron", "Linear SVM", "Logistic Regression", "LDA"
]

def build_linear_model(name, C=1.0):
    if name == "Linear SVM":
        return LinearSVC(C=C, max_iter=5000)
    if name == "Logistic Regression":
        return LogisticRegression(C=C, max_iter=2000)
    if name == "Perceptron":
        return Perceptron(max_iter=2000, tol=1e-3)
    if name == "LDA":
        return LinearDiscriminantAnalysis()
    return LogisticRegression(C=C, max_iter=2000)

def build_nonlinear_model(
    name,
    C=1.0, gamma=1.0, svm_degree=3,
    n_neighbors=7, weights='uniform',
    max_depth=6, min_samples_split=2,
    n_estimators=100,
    hidden_layer_sizes=50, mlp_alpha=0.0001, learning_rate_init=0.001
):
    if name == "SVM (RBF)":
        return SVC(kernel='rbf', probability=True, C=C, gamma=gamma)
    if name == "SVM (Poly)":
        return SVC(kernel='poly', probability=True, C=C, gamma=gamma, degree=svm_degree)
    if name == "Linear SVM":
        return SVC(kernel='linear', probability=True, C=C)
    if name == "KNN":
        return KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights)
    if name == "Decision Tree":
        return DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split)
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)
    if name == "AdaBoost":
        return AdaBoostClassifier(n_estimators=n_estimators)
    if name == "MLP Neural Network":
        return MLPClassifier(
            hidden_layer_sizes=(hidden_layer_sizes,),
            alpha=mlp_alpha,
            learning_rate_init=learning_rate_init,
            max_iter=1000
        )
    if name == "Naive Bayes":
        return GaussianNB()
    if name == "LDA":
        return LinearDiscriminantAnalysis()
    if name == "QDA":
        return QuadraticDiscriminantAnalysis()
    if name == "Perceptron":
        return Perceptron(max_iter=1000, tol=1e-3)
    if name == "Logistic Regression":
        return LogisticRegression(C=C, max_iter=2000)
    if name == "SGD (logistic)":
        return SGDClassifier(loss="log_loss", max_iter=2000, alpha=C)
    if name == "Gradient Boosting":
        return GradientBoostingClassifier()
    if name == "Extra Trees":
        return ExtraTreesClassifier(n_estimators=n_estimators, max_depth=max_depth)
    return SVC(kernel='rbf', probability=True, C=C, gamma=gamma)

def build_regression_model(name, alpha=1.0, n_estimators=100,
                           hidden_units=50, mlp_alpha=0.0001,
                           learning_rate_init=0.001):
    if name == "Linear Regression":
        return LinearRegression()
    if name == "Random Forest Regressor":
        return RandomForestRegressor(n_estimators=n_estimators, max_depth=6)
    if name == "Decision Tree Regressor":
        return DecisionTreeRegressor(max_depth=6)
    if name == "KNN Regressor":
        return KNeighborsRegressor(n_neighbors=7, weights='distance')
    if name == "MLP Regressor":
        return MLPRegressor(
            hidden_layer_sizes=(hidden_units,),
            alpha=mlp_alpha,
            learning_rate_init=learning_rate_init,
            max_iter=1000
        )
    if name == "Ridge Regression":
        return Ridge(alpha=alpha)
    if name == "Lasso Regression":
        return Lasso(alpha=alpha, max_iter=5000)
    if name == "ElasticNet Regression":
        return ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=5000)
    if name == "SVR (RBF)":
        return SVR(C=1.0, gamma='scale')
    return LinearRegression()

data_source_selector = Dropdown(
    options=["Synthetic data", "Uploaded CSV"],
    value="Synthetic data",
    description="Data src",
    layout=Layout(width="95%")
)

upload_widget = FileUpload(
    accept=".csv",
    multiple=False,
    description="Upload CSV",
    layout=Layout(width="95%")
)

use_pca_checkbox = Checkbox(
    value=True,
    description="Use PCA (2D proj)",
    indent=False
)

target_col_dropdown = Dropdown(
    options=[],
    description="Target col",
    layout=Layout(width="95%")
)

feature_cols_dropdown = Dropdown(
    options=[],
    description="Feature col",
    layout=Layout(width="95%")
)

csv_info_out = Output(
    layout=Layout(
        border="1px solid #444",
        width="95%",
        height="150px",
        overflow_y="auto",
        overflow_x="auto"
    )
)

csv_details_box = VBox([
    csv_info_out,
    target_col_dropdown,
    feature_cols_dropdown,
])

csv_accordion = Accordion(children=[csv_details_box])
csv_accordion.set_title(0, "CSV details & columns")

animate_checkbox = Checkbox(
    value=False,
    description="Show 3D animation",
    indent=False
)

anim_interval_slider = IntSlider(
    min=50, max=1000, step=50, value=200,
    description="Anim interval",
    layout=Layout(width="95%"),
    continuous_update=False
)

anim_frames_slider = IntSlider(
    min=12, max=60, step=4, value=24,
    description="Anim frames",
    layout=Layout(width="95%"),
    continuous_update=False
)

uploaded_df = {"df": None}

def on_upload_change(change):
    csv_info_out.clear_output()
    if not upload_widget.value:
        target_col_dropdown.options = []
        feature_cols_dropdown.options = []
        uploaded_df["df"] = None
        return

    key = list(upload_widget.value.keys())[0]
    content = upload_widget.value[key]["content"]
    df = pd.read_csv(io.BytesIO(content))
    uploaded_df["df"] = df

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        with csv_info_out:
            print("No numeric columns in CSV.")
        return

    target_col_dropdown.options = numeric_cols
    target_col_dropdown.value = numeric_cols[-1]

    feature_cols_dropdown.options = numeric_cols[:-1] or numeric_cols
    feature_cols_dropdown.value = feature_cols_dropdown.options[0]

    with csv_info_out:
        print(f"Rows: {df.shape[0]}, Cols: {df.shape[1]}")

        max_cols_display = 6
        shown = numeric_cols[:max_cols_display]
        more = len(numeric_cols) - max_cols_display
        msg = ", ".join(shown)
        if more > 0:
            msg += f" ... (+{more} more)"
        print("Numeric cols:", msg)

        print("Preview (head 3):")
        display(df.head(3))

upload_widget.observe(on_upload_change, names="value")

def on_data_source_change(change):
    is_csv = (change["new"] == "Uploaded CSV")
    upload_widget.layout.display = "block" if is_csv else "none"
    csv_accordion.layout.display = "block" if is_csv else "none"

data_source_selector.observe(on_data_source_change, names="value")
on_data_source_change({"new": data_source_selector.value})

def get_data_for_task(task, dataset_name, n_samples,
                      noise_pct, outliers_pct, random_state):
    if data_source_selector.value == "Uploaded CSV" and uploaded_df["df"] is not None:
        df = uploaded_df["df"]
        target_col = target_col_dropdown.value
        feature_col = feature_cols_dropdown.value
        if target_col is None or feature_col is None:
            raise ValueError("Please choose target & feature columns.")
        X = df[[feature_col]].values
        if X.shape[1] > 1:
            X = X[:, :1]

        rng = np.random.RandomState(random_state)
        X2 = rng.normal(scale=1.0, size=(X.shape[0], 1))
        X = np.hstack([X, X2])

        y = df[target_col].values

        if task in ["linear_cls", "nonlinear_cls"]:
            if len(np.unique(y)) > 2:
                median = np.median(y)
                y = (y > median).astype(int)
            else:
                uniques = np.unique(y)
                if set(uniques) != {0, 1}:
                    mapping = {uniques[0]: 0, uniques[1]: 1}
                    y = np.vectorize(mapping.get)(y)
            X, y = apply_noise_outliers_classification(
                X, y, noise_pct=noise_pct, outliers_pct=outliers_pct,
                random_state=random_state
            )
        else:
            X, y = apply_noise_outliers_regression(
                X, y, noise_pct=noise_pct, outliers_pct=outliers_pct,
                random_state=random_state
            )
        return X, y

    key = (task, dataset_name, n_samples, noise_pct, outliers_pct, random_state)
    if key in data_cache:
        return data_cache[key]

    noise_std = 0.2
    if task == "regression":
        noise_std = 0.01 + (noise_pct / 100.0) * 1.0

    if task == "linear_cls":
        X, y = generate_linear_data(dataset_name, n_samples, noise_std, random_state)
        X, y = apply_noise_outliers_classification(
            X, y, noise_pct=noise_pct, outliers_pct=outliers_pct,
            random_state=random_state
        )
    elif task == "nonlinear_cls":
        X, y = generate_nonlinear_data(dataset_name, n_samples, noise_std, random_state)
        X, y = apply_noise_outliers_classification(
            X, y, noise_pct=noise_pct, outliers_pct=outliers_pct,
            random_state=random_state
        )
    elif task == "regression":
        X, y = generate_regression_data(dataset_name, n_samples, noise_std, random_state)
        X, y = apply_noise_outliers_regression(
            X, y, noise_pct=noise_pct, outliers_pct=outliers_pct,
            random_state=random_state
        )
    else:
        raise ValueError("Unknown task")

    data_cache[key] = (X, y)
    return X, y

def maybe_pca_transform(X, task):
    if not use_pca_checkbox.value:
        return X
    if X.shape[1] <= 2:
        return X
    pca = PCA(n_components=2)
    return pca.fit_transform(X)

def plot_linear_classification(
    model_name, dataset, noise_pct, outliers_pct, n_samples, test_size,
    random_state, azimuth, elevation, alpha_points, point_size,
    C, colormap, boundary_mode, w1_manual, w2_manual, b_manual,
    animate, anim_interval, anim_frames
):
    try:
        X, y = get_data_for_task(
            task="linear_cls",
            dataset_name=dataset,
            n_samples=n_samples,
            noise_pct=noise_pct,
            outliers_pct=outliers_pct,
            random_state=random_state
        )
    except Exception as e:
        print("Data error:", e)
        return None

    X_vis = maybe_pca_transform(X, task="linear_cls")

    X_train, X_test, y_train, y_test = train_test_split(
        X_vis, y, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler = StandardScaler()
    model = build_linear_model(model_name, C=C)

    pipeline = make_pipeline(scaler, model)
    base_model = model

    scores_test = None
    train_acc = 0.0
    roc_auc = 0.0

    if boundary_mode == "Model":
        pipeline.fit(X_train, y_train)
        y_train_pred = pipeline.predict(X_train)
        y_test_pred = pipeline.predict(X_test)
        train_acc = np.mean(y_train_pred == y_train)

        fpr, tpr, roc_auc = compute_roc_auc(pipeline, X_test, y_test)

        if hasattr(pipeline, "decision_function"):
            try:
                scores_test = pipeline.decision_function(X_test)
            except Exception:
                scores_test = pipeline.predict_proba(X_test)[:, 1]
        elif hasattr(pipeline, "predict_proba"):
            scores_test = pipeline.predict_proba(X_test)[:, 1]
        else:
            scores_test = pipeline.predict(X_test).astype(float)

        cm = confusion_matrix(y_test, y_test_pred)

    else:
        scores_train = w1_manual * X_train[:, 0] + w2_manual * X_train[:, 1] + b_manual
        y_train_pred = (scores_train > 0).astype(int)
        scores_test = w1_manual * X_test[:, 0] + w2_manual * X_test[:, 1] + b_manual
        y_test_pred = (scores_test > 0).astype(int)
        train_acc = np.mean(y_train_pred == y_train)
        fpr, tpr, _ = roc_curve(y_test, scores_test)
        roc_auc = auc(fpr, tpr)
        cm = confusion_matrix(y_test, y_test_pred)
        pipeline = None
        base_model = None

    x_min, x_max = X_vis[:, 0].min() - 0.6, X_vis[:, 0].max() + 0.6
    y_min, y_max = X_vis[:, 1].min() - 0.6, X_vis[:, 1].max() + 0.6
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 100),
        np.linspace(y_min, y_max, 100)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    if boundary_mode == "Model":
        try:
            Z = pipeline.predict(grid).reshape(xx.shape)
        except Exception:
            Z = np.zeros(xx.shape)
        if hasattr(pipeline, "decision_function"):
            try:
                margin_vals = pipeline.decision_function(grid).reshape(xx.shape)
            except Exception:
                margin_vals = None
        else:
            margin_vals = None
        model_vals_surface = margin_vals if margin_vals is not None else Z.astype(float)
        manual_vals = w1_manual * xx + w2_manual * yy + b_manual
    else:
        scores_grid = w1_manual * grid[:, 0] + w2_manual * grid[:, 1] + b_manual
        Z = (scores_grid > 0).astype(int).reshape(xx.shape)
        margin_vals = w1_manual * xx + w2_manual * yy + b_manual
        model_vals_surface = margin_vals
        manual_vals = None

    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(3, 3, height_ratios=[0.6, 2, 2], figure=fig)

    ax_sum = fig.add_subplot(gs[0, :])
    summary_lines = [
        f"Data source : {data_source_selector.value}   |   Dataset: {dataset}",
        f"Model      : {model_name}   |   samples: {len(X)} (features: {X.shape[1]})",
        f"Noise %    : {noise_pct:.1f}   |   Outliers %: {outliers_pct:.1f}   |   "
        f"Test size: {test_size:.2f}   |   Rand seed: {random_state}",
        f"Train acc  : {train_acc:.3f}   |   AUC: {roc_auc:.3f}   |   Mode: {boundary_mode}"
    ]
    draw_summary_box(ax_sum, summary_lines)

    ax1 = fig.add_subplot(gs[1, 0])
    ax1.contourf(xx, yy, Z, alpha=0.35, cmap=colormap, levels=np.linspace(0, 1, 3))
    ax1.scatter(X_vis[:, 0], X_vis[:, 1], c=y, s=point_size,
                cmap=colormap, edgecolors="k", alpha=alpha_points)
    ax1.set_title("Decision regions", fontsize=10)

    if boundary_mode == "Model":
        if margin_vals is not None:
            ax1.contour(xx, yy, margin_vals, levels=[-1, 1],
                        colors="black", linestyles="--", linewidths=2)
        if manual_vals is not None:
            ax1.contour(xx, yy, manual_vals, levels=[0],
                        colors="yellow", linestyles="--", linewidths=2)
    else:
        if margin_vals is not None:
            ax1.contour(xx, yy, margin_vals, levels=[0],
                        colors="yellow", linestyles="--", linewidths=2)

    ax2 = fig.add_subplot(gs[1, 1], projection="3d")
    ax2.plot_surface(xx, yy, model_vals_surface, rstride=1, cstride=1,
                     alpha=alpha_points, cmap=colormap, edgecolor="k")
    ax2.scatter(X_vis[:, 0], X_vis[:, 1], y, c=y, s=point_size,
                cmap=colormap, edgecolors="k", alpha=alpha_points)
    ax2.set_xlabel("X1")
    ax2.set_ylabel("X2")
    ax2.set_zlabel("Score / Pred")
    ax2.set_title("3D decision surface", fontsize=10)
    ax2.view_init(elev=elevation, azim=azimuth)

    ax3 = fig.add_subplot(gs[1, 2])
    ax3.plot(fpr, tpr, lw=3, label=f"AUC = {roc_auc:.3f}")
    ax3.plot([0, 1], [0, 1], "--", color="gray", label="random")
    ax3.set_xlabel("FPR")
    ax3.set_ylabel("TPR")
    ax3.set_title("ROC Curve (test set)", fontsize=10)
    ax3.legend()
    ax3.grid(alpha=0.25)

    ax4 = fig.add_subplot(gs[2, 0])
    ax4.imshow(cm, interpolation="nearest", cmap="Blues")
    ax4.set_title("Confusion matrix", fontsize=10)
    ax4.set_xlabel("Predicted label")
    ax4.set_ylabel("True label")
    for (i, j), v in np.ndenumerate(cm):
        ax4.text(j, i, str(v), ha="center", va="center", color="black", fontsize=8)

    ax5 = fig.add_subplot(gs[2, 1])
    if pipeline is not None:
        plot_learning_curve_ax(
            ax5,
            pipeline,
            X_vis,
            y,
            task_type="cls",
            random_state=random_state
        )
    else:
        ax5.text(
            0.5, 0.5,
            "Learning curve\nnot available in Manual mode",
            ha="center", va="center", fontsize=9
        )
        ax5.set_axis_off()

    ax6 = fig.add_subplot(gs[2, 2])

    if pipeline is not None and base_model is not None:
        family = get_model_family(base_model)

        if family == "linear" and hasattr(base_model, "coef_"):
            coefs = base_model.coef_.ravel()
            ax6.bar(range(len(coefs)), coefs)
            ax6.set_title("Coefficients", fontsize=10)
            ax6.set_xlabel("Feature index")
            ax6.set_ylabel("Weight")

        elif hasattr(base_model, "feature_importances_"):
            imps = base_model.feature_importances_
            ax6.bar(range(len(imps)), imps)
            ax6.set_title("Feature importances", fontsize=10)
            ax6.set_xlabel("Feature index")
            ax6.set_ylabel("Importance")

        else:
            ax6.hist(scores_test[y_test == 0], bins=15, alpha=0.7, label="class 0")
            ax6.hist(scores_test[y_test == 1], bins=15, alpha=0.7, label="class 1")
            ax6.set_title("Score distribution (test)", fontsize=10)
            ax6.set_xlabel("Score")
            ax6.set_ylabel("Count")
            ax6.legend()
    else:
        ax6.hist(scores_test[y_test == 0], bins=15, alpha=0.7, label="class 0")
        ax6.hist(scores_test[y_test == 1], bins=15, alpha=0.7, label="class 1")
        ax6.set_title("Score distribution (test)", fontsize=10)
        ax6.set_xlabel("Score")
        ax6.set_ylabel("Count")
        ax6.legend()

    plt.tight_layout()
    plt.show()

    if animate:
        fig2 = plt.figure(figsize=(6, 5))
        ax_anim = fig2.add_subplot(111, projection="3d")
        surf = ax_anim.plot_surface(xx, yy, model_vals_surface, rstride=1, cstride=1,
                                    alpha=alpha_points, cmap=colormap, edgecolor="k")
        ax_anim.scatter(X_vis[:, 0], X_vis[:, 1], y, c=y, s=point_size,
                        cmap=colormap, edgecolors="k", alpha=alpha_points)
        ax_anim.set_xlabel("X1")
        ax_anim.set_ylabel("X2")
        ax_anim.set_zlabel("Score / Pred")
        ax_anim.set_title("3D rotating view", fontsize=10)

        def update(frame):
            ax_anim.view_init(elev=elevation, azim=frame)
            return surf,

        frames = np.linspace(0, 360, anim_frames)
        ani = animation.FuncAnimation(
            fig2, update, frames=frames, interval=anim_interval, blit=False
        )
        plt.close(fig2)

        html_anim = ani.to_jshtml()
        wrapper = f"""
        <div style="
            width: 100%;
            display: flex;
            justify-content: center;
        ">
          <div style="max-width: 900px; flex: 1 1 auto;">
            {html_anim}
          </div>
        </div>
        """
        display(HTML(wrapper))

    return fig


def plot_nonlinear_classification(
    model_name, dataset, poly_degree, noise_pct, outliers_pct, n_samples, test_size,
    random_state, azimuth, elevation, alpha_points, point_size,
    colormap, C, gamma, svm_degree,
    n_neighbors, weights,
    max_depth, min_samples_split,
    n_estimators,
    hidden_layer_sizes, mlp_alpha, learning_rate_init,
    animate, anim_interval, anim_frames
):
    try:
        X, y = get_data_for_task(
            task="nonlinear_cls",
            dataset_name=dataset,
            n_samples=n_samples,
            noise_pct=noise_pct,
            outliers_pct=outliers_pct,
            random_state=random_state
        )
    except Exception as e:
        print("Data error:", e)
        return None

    X_vis = maybe_pca_transform(X, task="nonlinear_cls")

    X_train, X_test, y_train, y_test = train_test_split(
        X_vis, y, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler = StandardScaler()
    if poly_degree > 1:
        poly = PolynomialFeatures(degree=poly_degree, include_bias=False)
        base_model = build_nonlinear_model(
            model_name,
            C=C, gamma=gamma, svm_degree=svm_degree,
            n_neighbors=n_neighbors, weights=weights,
            max_depth=max_depth, min_samples_split=min_samples_split,
            n_estimators=n_estimators,
            hidden_layer_sizes=hidden_layer_sizes, mlp_alpha=mlp_alpha,
            learning_rate_init=learning_rate_init
        )
        pipeline = make_pipeline(poly, scaler, base_model)
    else:
        base_model = build_nonlinear_model(
            model_name,
            C=C, gamma=gamma, svm_degree=svm_degree,
            n_neighbors=n_neighbors, weights=weights,
            max_depth=max_depth, min_samples_split=min_samples_split,
            n_estimators=n_estimators,
            hidden_layer_sizes=hidden_layer_sizes, mlp_alpha=mlp_alpha,
            learning_rate_init=learning_rate_init
        )
        pipeline = make_pipeline(scaler, base_model)

    pipeline.fit(X_train, y_train)
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    train_acc = np.mean(y_train_pred == y_train)

    fpr, tpr, roc_auc = compute_roc_auc(pipeline, X_test, y_test)

    if hasattr(pipeline, "decision_function"):
        try:
            scores_test = pipeline.decision_function(X_test)
        except Exception:
            scores_test = pipeline.predict_proba(X_test)[:, 1]
    elif hasattr(pipeline, "predict_proba"):
        scores_test = pipeline.predict_proba(X_test)[:, 1]
    else:
        scores_test = pipeline.predict(X_test).astype(float)

    cm = confusion_matrix(y_test, y_test_pred)

    x_min, x_max = X_vis[:, 0].min() - 0.6, X_vis[:, 0].max() + 0.6
    y_min, y_max = X_vis[:, 1].min() - 0.6, X_vis[:, 1].max() + 0.6
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 100),
        np.linspace(y_min, y_max, 100)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    try:
        Z = pipeline.predict(grid).reshape(xx.shape)
    except Exception:
        try:
            Z = pipeline.predict_proba(grid)[:, 1].reshape(xx.shape)
        except Exception:
            Z = np.zeros(xx.shape)

    margin_vals = None
    if model_name in ["SVM (RBF)", "SVM (Poly)", "Linear SVM"]:
        try:
            margin_vals = pipeline.decision_function(grid).reshape(xx.shape)
        except Exception:
            margin_vals = None

    family = get_model_family(base_model)

    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(3, 3, height_ratios=[0.6, 2, 2], figure=fig)

    ax_sum = fig.add_subplot(gs[0, :])
    summary_lines = [
        f"Data source : {data_source_selector.value}   |   Dataset: {dataset}",
        f"Model      : {model_name}   |   samples: {len(X)} (features: {X.shape[1]})",
        f"Noise %    : {noise_pct:.1f}   |   Outliers %: {outliers_pct:.1f}   |   "
        f"Test size: {test_size:.2f}   |   Rand seed: {random_state}",
        f"Train acc  : {train_acc:.3f}   |   AUC: {roc_auc:.3f}"
    ]
    draw_summary_box(ax_sum, summary_lines)

    ax1 = fig.add_subplot(gs[1, 0])
    ax1.contourf(xx, yy, Z, alpha=0.35, cmap=colormap, levels=np.linspace(0, 1, 3))
    ax1.scatter(X_vis[:, 0], X_vis[:, 1], c=y, s=point_size,
                cmap=colormap, edgecolors="k", alpha=alpha_points)
    ax1.set_title(f"{model_name} | {dataset} | degree={poly_degree}", fontsize=10)

    if family == "svm" and hasattr(base_model, "support_"):
        sv_idx = base_model.support_
        sv_points = X_train[sv_idx]
        ax1.scatter(
            sv_points[:, 0],
            sv_points[:, 1],
            s=point_size * 1.8,
            facecolors="none",
            edgecolors="yellow",
            linewidths=1.5,
            label="SV"
        )
        ax1.legend(fontsize=8)

    if margin_vals is not None:
        ax1.contour(xx, yy, margin_vals, levels=[-1, 1],
                    colors='yellow', linestyles='--', linewidths=2)

    ax2 = fig.add_subplot(gs[1, 1], projection="3d")
    ax2.plot_surface(xx, yy, Z.astype(float), rstride=1, cstride=1,
                     alpha=alpha_points, cmap=colormap, edgecolor="k")
    ax2.scatter(X_vis[:, 0], X_vis[:, 1], y, c=y, s=point_size,
                cmap=colormap, edgecolors="k", alpha=alpha_points)
    ax2.set_xlabel("X1")
    ax2.set_ylabel("X2")
    ax2.set_zlabel("Prediction/Prob")
    ax2.set_title("3D decision surface", fontsize=10)
    ax2.view_init(elev=elevation, azim=azimuth)

    ax3 = fig.add_subplot(gs[1, 2])
    ax3.plot(fpr, tpr, lw=3, label=f"AUC = {roc_auc:.3f}")
    ax3.plot([0, 1], [0, 1], "--", color="gray", label="random")
    ax3.set_xlabel("False Positive Rate")
    ax3.set_ylabel("True Positive Rate")
    ax3.set_title("ROC Curve (test set)", fontsize=10)
    ax3.legend()
    ax3.grid(alpha=0.25)

    ax4 = fig.add_subplot(gs[2, 0])
    ax4.imshow(cm, interpolation="nearest", cmap="Blues")
    ax4.set_title("Confusion matrix", fontsize=10)
    ax4.set_xlabel("Predicted label")
    ax4.set_ylabel("True label")
    for (i, j), v in np.ndenumerate(cm):
        ax4.text(j, i, str(v), ha="center", va="center", color="black", fontsize=8)

    ax5 = fig.add_subplot(gs[2, 1])
    plot_learning_curve_ax(
        ax5,
        pipeline,
        X_vis,
        y,
        task_type="cls",
        random_state=random_state
    )

    ax6 = fig.add_subplot(gs[2, 2])

    if family == "linear" and hasattr(base_model, "coef_"):
        coefs = base_model.coef_.ravel()
        ax6.bar(range(len(coefs)), coefs)
        ax6.set_title("Coefficients", fontsize=10)
        ax6.set_xlabel("Feature index")
        ax6.set_ylabel("Weight")

    elif family == "tree_ensemble" and hasattr(base_model, "feature_importances_"):
        imps = base_model.feature_importances_
        ax6.bar(range(len(imps)), imps)
        ax6.set_title("Feature importances", fontsize=10)
        ax6.set_xlabel("Feature index")
        ax6.set_ylabel("Importance")

    elif family == "knn":
        k0 = base_model.n_neighbors
        ks = list(range(max(1, k0 - 4), k0 + 5))
        errors = []
        for k in ks:
            knn_tmp = KNeighborsClassifier(n_neighbors=k, weights=base_model.weights)
            pipe_tmp = make_pipeline(StandardScaler(), knn_tmp)
            pipe_tmp.fit(X_train, y_train)
            y_val_pred = pipe_tmp.predict(X_test)
            err = 1 - np.mean(y_val_pred == y_test)
            errors.append(err)
        ax6.plot(ks, errors, "o-")
        ax6.set_title("Error vs K", fontsize=10)
        ax6.set_xlabel("K")
        ax6.set_ylabel("Error")

    elif family == "mlp" and hasattr(base_model, "loss_curve_"):
        losses = base_model.loss_curve_
        ax6.plot(range(1, len(losses) + 1), losses, "o-")
        ax6.set_title("MLP loss curve", fontsize=10)
        ax6.set_xlabel("Epoch")
        ax6.set_ylabel("Loss")

    else:
        ax6.hist(scores_test[y_test == 0], bins=15, alpha=0.7, label="class 0")
        ax6.hist(scores_test[y_test == 1], bins=15, alpha=0.7, label="class 1")
        ax6.set_title("Score distribution (test)", fontsize=10)
        ax6.set_xlabel("Score")
        ax6.set_ylabel("Count")
        ax6.legend()

    plt.tight_layout()
    plt.show()

    if animate:
        fig2 = plt.figure(figsize=(6, 5))
        ax_anim = fig2.add_subplot(111, projection="3d")
        surf = ax_anim.plot_surface(xx, yy, Z.astype(float), rstride=1, cstride=1,
                                    alpha=alpha_points, cmap=colormap, edgecolor="k")
        ax_anim.scatter(X_vis[:, 0], X_vis[:, 1], y, c=y, s=point_size,
                        cmap=colormap, edgecolors="k", alpha=alpha_points)
        ax_anim.set_xlabel("X1")
        ax_anim.set_ylabel("X2")
        ax_anim.set_zlabel("Prediction/Prob")
        ax_anim.set_title("3D rotating view", fontsize=10)

        def update(frame):
            ax_anim.view_init(elev=elevation, azim=frame)
            return surf,

        frames = np.linspace(0, 360, anim_frames)
        ani = animation.FuncAnimation(
            fig2, update, frames=frames, interval=anim_interval, blit=False
        )
        plt.close(fig2)

        html_anim = ani.to_jshtml()
        wrapper = f"""
        <div style="
            width: 100%;
            display: flex;
            justify-content: center;
        ">
          <div style="max-width: 900px; flex: 1 1 auto;">
            {html_anim}
          </div>
        </div>
        """
        display(HTML(wrapper))

    return fig


def plot_regression(
    model_name, dataset, noise_pct, outliers_pct, n_samples, test_size,
    random_state, azimuth, elevation, alpha_points, point_size,
    colormap, alpha_reg, n_estimators, hidden_units, mlp_alpha, learning_rate_init,
    animate, anim_interval, anim_frames
):
    try:
        X, y = get_data_for_task(
            task="regression",
            dataset_name=dataset,
            n_samples=n_samples,
            noise_pct=noise_pct,
            outliers_pct=outliers_pct,
            random_state=random_state
        )
    except Exception as e:
        print("Data error:", e)
        return None

    X_vis = maybe_pca_transform(X, task="regression")

    X_train, X_test, y_train, y_test = train_test_split(
        X_vis, y, test_size=test_size, random_state=random_state
    )

    scaler = StandardScaler()
    model = build_regression_model(
        model_name,
        alpha=alpha_reg,
        n_estimators=n_estimators,
        hidden_units=hidden_units,
        mlp_alpha=mlp_alpha,
        learning_rate_init=learning_rate_init
    )
    pipeline = make_pipeline(scaler, model)
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    x_min, x_max = X_vis[:, 0].min() - 0.6, X_vis[:, 0].max() + 0.6
    y_min, y_max = X_vis[:, 1].min() - 0.6, X_vis[:, 1].max() + 0.6
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 50),
        np.linspace(y_min, y_max, 50)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    z_grid = pipeline.predict(grid).reshape(xx.shape)

    base_model = pipeline[-1]
    family = get_reg_model_family(base_model)

    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(3, 3, height_ratios=[0.6, 2, 2], figure=fig)

    ax_sum = fig.add_subplot(gs[0, :])
    summary_lines = [
        f"Data source : {data_source_selector.value}   |   Dataset: {dataset}",
        f"Model      : {model_name}   |   samples: {len(X)} (features: {X.shape[1]})",
        f"Noise %    : {noise_pct:.1f}   |   Outliers %: {outliers_pct:.1f}   |   "
        f"Test size: {test_size:.2f}   |   Rand seed: {random_state}",
        f"MSE        : {mse:.3f}   |   R²: {r2:.3f}"
    ]
    draw_summary_box(ax_sum, summary_lines)

    ax1 = fig.add_subplot(gs[1, 0])
    sc = ax1.scatter(X_vis[:, 0], X_vis[:, 1], c=y, s=point_size,
                     cmap=colormap, edgecolors="k", alpha=alpha_points)
    ax1.set_title(f"{model_name} | {dataset}", fontsize=10)
    ax1.set_xlabel("X1")
    ax1.set_ylabel("X2")
    plt.colorbar(sc, ax=ax1, label="y")

    ax2 = fig.add_subplot(gs[1, 1], projection="3d")
    surf = ax2.plot_surface(xx, yy, z_grid, rstride=1, cstride=1,
                            alpha=alpha_points, cmap=colormap, edgecolor="k")
    ax2.scatter(X_vis[:, 0], X_vis[:, 1], y, c=y, cmap=colormap,
                s=point_size, edgecolors="k", alpha=alpha_points)
    ax2.set_xlabel("X1")
    ax2.set_ylabel("X2")
    ax2.set_zlabel("y / pred")
    ax2.set_title("Regression surface", fontsize=10)
    ax2.view_init(elev=elevation, azim=azimuth)

    ax3 = fig.add_subplot(gs[1, 2])
    ax3.scatter(y_test, y_pred, alpha=0.7, s=20, edgecolors="k")
    min_v = min(y_test.min(), y_pred.min())
    max_v = max(y_test.max(), y_pred.max())
    ax3.plot([min_v, max_v], [min_v, max_v], "r--", lw=2)
    ax3.set_xlabel("True y")
    ax3.set_ylabel("Predicted y")
    ax3.set_title(f"MSE={mse:.3f} | R²={r2:.3f}", fontsize=10)

    ax4 = fig.add_subplot(gs[2, 0])
    residuals = y_pred - y_test
    ax4.hist(residuals, bins=20, alpha=0.8)
    ax4.set_title("Prediction error histogram", fontsize=10)
    ax4.set_xlabel("y_pred - y_true")
    ax4.set_ylabel("Count")

    ax5 = fig.add_subplot(gs[2, 1])
    plot_learning_curve_ax(
        ax5,
        pipeline,
        X_vis,
        y,
        task_type="reg",
        random_state=random_state
    )

    ax6 = fig.add_subplot(gs[2, 2])

    if family == "linear" and hasattr(base_model, "coef_"):
        coefs = base_model.coef_.ravel()
        ax6.bar(range(len(coefs)), coefs)
        ax6.set_title("Coefficients", fontsize=10)
        ax6.set_xlabel("Feature index")
        ax6.set_ylabel("Weight")

    elif family == "tree_ensemble" and hasattr(base_model, "feature_importances_"):
        imps = base_model.feature_importances_
        ax6.bar(range(len(imps)), imps)
        ax6.set_title("Feature importances", fontsize=10)
        ax6.set_xlabel("Feature index")
        ax6.set_ylabel("Importance")

    elif family == "knn":
        base_k = base_model.n_neighbors
        ks = list(range(max(1, base_k - 4), base_k + 5))
        errors = []
        for k in ks:
            knn_tmp = KNeighborsRegressor(n_neighbors=k, weights=base_model.weights)
            pipe_tmp = make_pipeline(StandardScaler(), knn_tmp)
            pipe_tmp.fit(X_train, y_train)
            y_val_pred = pipe_tmp.predict(X_test)
            err = mean_squared_error(y_test, y_val_pred)
            errors.append(err)
        ax6.plot(ks, errors, "o-")
        ax6.set_title("MSE vs K", fontsize=10)
        ax6.set_xlabel("K")
        ax6.set_ylabel("MSE")

    elif family == "mlp" and hasattr(base_model, "loss_curve_"):
        losses = base_model.loss_curve_
        ax6.plot(range(1, len(losses) + 1), losses, "o-")
        ax6.set_title("MLP loss curve", fontsize=10)
        ax6.set_xlabel("Epoch")
        ax6.set_ylabel("Loss")

    else:
        ax6.text(
            0.5, 0.5,
            "No extra diagnostics",
            ha="center", va="center", fontsize=9
        )
        ax6.set_axis_off()

    plt.tight_layout()
    plt.show()

    if animate:
        fig2 = plt.figure(figsize=(6, 5))
        ax_anim = fig2.add_subplot(111, projection="3d")
        surf2 = ax_anim.plot_surface(xx, yy, z_grid, rstride=1, cstride=1,
                                     alpha=alpha_points, cmap=colormap, edgecolor="k")
        ax_anim.scatter(X_vis[:, 0], X_vis[:, 1], y, c=y, cmap=colormap,
                        s=point_size, edgecolors="k", alpha=alpha_points)
        ax_anim.set_xlabel("X1")
        ax_anim.set_ylabel("X2")
        ax_anim.set_zlabel("y / pred")
        ax_anim.set_title("3D rotating regression surface", fontsize=10)

        def update(frame):
            ax_anim.view_init(elev=elevation, azim=frame)
            return surf2,

        frames = np.linspace(0, 360, anim_frames)
        ani = animation.FuncAnimation(
            fig2, update, frames=frames, interval=anim_interval, blit=False
        )
        plt.close(fig2)

        html_anim = ani.to_jshtml()
        wrapper = f"""
        <div style="
            width: 100%;
            display: flex;
            justify-content: center;
        ">
          <div style="max-width: 900px; flex: 1 1 auto;">
            {html_anim}
          </div>
        </div>
        """
        display(HTML(wrapper))

    return fig

model_selector_lin = Dropdown(
    options=linear_model_names,
    value="Linear SVM",
    description="Model",
    layout=Layout(width="95%")
)

dataset_selector_lin = Dropdown(
    options=["blobs", "shifted", "vertical"],
    value="blobs",
    description="Dataset",
    layout=Layout(width="95%")
)

boundary_mode_selector_lin = Dropdown(
    options=["Model", "Manual"],
    value="Model",
    description="Boundary",
    layout=Layout(width="95%")
)

w1_slider_lin = FloatSlider(min=-5, max=5, step=0.1, value=1.0, description="w1", continuous_update=False)
w2_slider_lin = FloatSlider(min=-5, max=5, step=0.1, value=1.0, description="w2", continuous_update=False)
b_slider_lin  = FloatSlider(min=-5, max=5, step=0.1, value=0.0, description="b", continuous_update=False)

C_slider_lin = FloatSlider(min=0.01, max=10.0, step=0.01, value=1.0, description="C", continuous_update=False)

colormap_selector_lin = Dropdown(
    options=['coolwarm', 'bwr', 'viridis', 'plasma', 'cividis'],
    value='coolwarm',
    description='Colormap',
    layout=Layout(width="95%")
)

n_samples_slider_lin = IntSlider(min=100, max=2000, step=50, value=300, description="Samples", continuous_update=False)
noise_slider_lin = FloatSlider(min=0, max=100, step=1, value=10, description="Noise %", continuous_update=False)
outliers_slider_lin = FloatSlider(min=0, max=100, step=1, value=0, description="Outliers %", continuous_update=False)
test_size_slider_lin = FloatSlider(min=0.1, max=0.5, step=0.05, value=0.3, description="Test size", continuous_update=False)
random_state_slider_lin = IntSlider(min=0, max=9999, step=1, value=42, description="Rand seed", continuous_update=False)

azimuth_slider_lin   = IntSlider(min=0, max=360, value=45, description="Azimuth", continuous_update=False)
elevation_slider_lin = IntSlider(min=0, max=90,  value=30, description="Elevation", continuous_update=False)
alpha_slider_lin     = FloatSlider(min=0.1, max=1.0, value=0.6, step=0.05, description="Alpha", continuous_update=False)
point_size_slider_lin = IntSlider(min=5, max=100, value=20, description="Point size", continuous_update=False)

advanced_box_lin = VBox([test_size_slider_lin, random_state_slider_lin])
advanced_acc_lin = Accordion(children=[advanced_box_lin])
advanced_acc_lin.set_title(0, "Advanced (test size / seed)")

reset_linear_btn = Button(description="Reset linear", button_style="warning", layout=Layout(width="95%"))
update_linear_btn = Button(description="Update linear plot", button_style="primary", layout=Layout(width="95%"))

def reset_linear(_):
    model_selector_lin.value = "Linear SVM"
    dataset_selector_lin.value = "blobs"
    boundary_mode_selector_lin.value = "Model"
    w1_slider_lin.value = 1.0
    w2_slider_lin.value = 1.0
    b_slider_lin.value = 0.0
    C_slider_lin.value = 1.0
    colormap_selector_lin.value = "coolwarm"
    n_samples_slider_lin.value = 300
    noise_slider_lin.value = 10
    outliers_slider_lin.value = 0
    test_size_slider_lin.value = 0.3
    random_state_slider_lin.value = 42
    azimuth_slider_lin.value = 45
    elevation_slider_lin.value = 30
    alpha_slider_lin.value = 0.6
    point_size_slider_lin.value = 20

reset_linear_btn.on_click(reset_linear)

linear_controls = VBox([
    Label("Linear models"),
    model_selector_lin,
    dataset_selector_lin,
    boundary_mode_selector_lin,
    w1_slider_lin,
    w2_slider_lin,
    b_slider_lin,
    C_slider_lin,
    colormap_selector_lin,
    n_samples_slider_lin,
    noise_slider_lin,
    outliers_slider_lin,
    advanced_acc_lin,
    azimuth_slider_lin,
    elevation_slider_lin,
    alpha_slider_lin,
    point_size_slider_lin,
    update_linear_btn,
    reset_linear_btn
])

model_selector_nl = Dropdown(
    options=[
        "SVM (RBF)", "SVM (Poly)", "Linear SVM",
        "KNN", "Decision Tree", "Random Forest",
        "AdaBoost", "MLP Neural Network",
        "Naive Bayes", "LDA", "QDA", "Perceptron", "Logistic Regression",
        "SGD (logistic)", "Gradient Boosting", "Extra Trees"
    ],
    value="SVM (RBF)",
    description="Model",
    layout=Layout(width="95%")
)

dataset_selector_nl = Dropdown(
    options=["moons", "circles", "blobs", "xor", "spiral", "checkerboard"],
    value="moons",
    description="Dataset",
    layout=Layout(width="95%")
)

poly_degree_slider_nl = IntSlider(min=1, max=20, value=3, description="Poly degree", continuous_update=False)

colormap_selector_nl = Dropdown(
    options=['coolwarm', 'bwr', 'viridis', 'plasma', 'cividis'],
    value='coolwarm',
    description='Colormap',
    layout=Layout(width="95%")
)

n_samples_slider_nl = IntSlider(min=100, max=2000, step=50, value=300, description="Samples", continuous_update=False)
noise_slider_nl = FloatSlider(min=0, max=100, step=1, value=10, description="Noise %", continuous_update=False)
outliers_slider_nl = FloatSlider(min=0, max=100, step=1, value=0, description="Outliers %", continuous_update=False)
test_size_slider_nl = FloatSlider(min=0.1, max=0.5, step=0.05, value=0.3, description="Test size", continuous_update=False)
random_state_slider_nl = IntSlider(min=0, max=9999, step=1, value=42, description="Rand seed", continuous_update=False)

azimuth_slider_nl   = IntSlider(min=0, max=360, value=45, description="Azimuth", continuous_update=False)
elevation_slider_nl = IntSlider(min=0, max=90,  value=30, description="Elevation", continuous_update=False)
alpha_slider_nl     = FloatSlider(min=0.1, max=1.0, value=0.6, step=0.05, description="Alpha", continuous_update=False)
point_size_slider_nl = IntSlider(min=5, max=100, value=20, description="Point size", continuous_update=False)

C_slider_nl       = FloatSlider(min=0.01, max=10.0, value=1.0, step=0.01, description='C', continuous_update=False)
gamma_slider_nl   = FloatSlider(min=0.01, max=5.0, value=1.0, step=0.01, description='Gamma', continuous_update=False)
svm_degree_slider = IntSlider(min=2, max=10, value=3, description='SVM degree', continuous_update=False)

n_neighbors_slider = IntSlider(min=1, max=20, value=7, description='KNN neighbors', continuous_update=False)
weights_knn_dropdown = Dropdown(options=['uniform', 'distance'], value='uniform', description='Weight')

max_depth_slider = IntSlider(min=1, max=20, value=6, description='Max depth', continuous_update=False)
min_samples_split_slider = IntSlider(min=2, max=20, value=2, description='Min split', continuous_update=False)

n_estimators_slider = IntSlider(min=10, max=500, value=100, description='N estimators', continuous_update=False)

hidden_units_slider = IntSlider(min=5, max=200, value=50, description='Hidden units', continuous_update=False)
mlp_alpha_slider    = FloatSlider(min=1e-5, max=0.1, value=0.0001, step=1e-4, description='MLP alpha', continuous_update=False)
learning_rate_slider = FloatSlider(min=1e-4, max=0.1, value=0.001, step=1e-4, description='LR init', continuous_update=False)

widgets_dict_nl = {
    "SVM (RBF)": [C_slider_nl, gamma_slider_nl],
    "SVM (Poly)": [C_slider_nl, gamma_slider_nl, svm_degree_slider],
    "Linear SVM": [C_slider_nl],
    "KNN": [n_neighbors_slider, weights_knn_dropdown],
    "Decision Tree": [max_depth_slider, min_samples_split_slider],
    "Random Forest": [n_estimators_slider, max_depth_slider],
    "AdaBoost": [n_estimators_slider],
    "MLP Neural Network": [hidden_units_slider, mlp_alpha_slider, learning_rate_slider],
    "Naive Bayes": [],
    "LDA": [],
    "QDA": [],
    "Perceptron": [],
    "Logistic Regression": [C_slider_nl],
    "SGD (logistic)": [C_slider_nl],
    "Gradient Boosting": [],
    "Extra Trees": [n_estimators_slider, max_depth_slider],
}

params_box_nl = VBox([])

def update_params_nl(*args):
    model = model_selector_nl.value
    params_box_nl.children = widgets_dict_nl.get(model, [])

model_selector_nl.observe(update_params_nl, names='value')
update_params_nl()

advanced_box_nl = VBox([test_size_slider_nl, random_state_slider_nl])
advanced_acc_nl = Accordion(children=[advanced_box_nl])
advanced_acc_nl.set_title(0, "Advanced (test size / seed)")

reset_nonlinear_btn = Button(description="Reset nonlinear", button_style="warning", layout=Layout(width="95%"))
update_nonlinear_btn = Button(description="Update nonlinear plot", button_style="primary", layout=Layout(width="95%"))

def reset_nonlinear(_):
    model_selector_nl.value = "SVM (RBF)"
    dataset_selector_nl.value = "moons"
    poly_degree_slider_nl.value = 3
    colormap_selector_nl.value = "coolwarm"
    n_samples_slider_nl.value = 300
    noise_slider_nl.value = 10
    outliers_slider_nl.value = 0
    test_size_slider_nl.value = 0.3
    random_state_slider_nl.value = 42
    azimuth_slider_nl.value = 45
    elevation_slider_nl.value = 30
    alpha_slider_nl.value = 0.6
    point_size_slider_nl.value = 20
    C_slider_nl.value = 1.0
    gamma_slider_nl.value = 1.0
    svm_degree_slider.value = 3
    n_neighbors_slider.value = 7
    weights_knn_dropdown.value = "uniform"
    max_depth_slider.value = 6
    min_samples_split_slider.value = 2
    n_estimators_slider.value = 100
    hidden_units_slider.value = 50
    mlp_alpha_slider.value = 0.0001
    learning_rate_slider.value = 0.001

reset_nonlinear_btn.on_click(reset_nonlinear)

nonlinear_controls = VBox([
    Label("Nonlinear models"),
    model_selector_nl,
    dataset_selector_nl,
    poly_degree_slider_nl,
    params_box_nl,
    colormap_selector_nl,
    n_samples_slider_nl,
    noise_slider_nl,
    outliers_slider_nl,
    advanced_acc_nl,
    azimuth_slider_nl,
    elevation_slider_nl,
    alpha_slider_nl,
    point_size_slider_nl,
    update_nonlinear_btn,
    reset_nonlinear_btn
])

reg_model_selector = Dropdown(
    options=[
        "Linear Regression",
        "Random Forest Regressor",
        "Decision Tree Regressor",
        "KNN Regressor",
        "MLP Regressor",
        "Ridge Regression",
        "Lasso Regression",
        "ElasticNet Regression",
        "SVR (RBF)"
    ],
    value="Linear Regression",
    description="Model",
    layout=Layout(width="95%")
)

reg_dataset_selector = Dropdown(
    options=["linear", "quadratic", "interaction"],
    value="linear",
    description="Dataset",
    layout=Layout(width="95%")
)

reg_colormap_selector = Dropdown(
    options=['viridis', 'plasma', 'cividis', 'coolwarm'],
    value='viridis',
    description='Colormap',
    layout=Layout(width="95%")
)

reg_n_samples_slider = IntSlider(min=100, max=2000, step=50, value=300, description="Samples", continuous_update=False)
reg_noise_slider = FloatSlider(min=0, max=100, step=1, value=10, description="Noise %", continuous_update=False)
reg_outliers_slider = FloatSlider(min=0, max=100, step=1, value=0, description="Outliers %", continuous_update=False)
reg_test_size_slider = FloatSlider(min=0.1, max=0.5, step=0.05, value=0.3, description="Test size", continuous_update=False)
reg_random_state_slider = IntSlider(min=0, max=9999, step=1, value=42, description="Rand seed", continuous_update=False)

reg_azimuth_slider   = IntSlider(min=0, max=360, value=45, description="Azimuth", continuous_update=False)
reg_elevation_slider = IntSlider(min=0, max=90,  value=30, description="Elevation", continuous_update=False)
reg_alpha_slider     = FloatSlider(min=0.1, max=1.0, value=0.6, step=0.05, description="Alpha", continuous_update=False)
reg_point_size_slider = IntSlider(min=5, max=100, value=20, description="Point size", continuous_update=False)

reg_alpha_reg_slider = FloatSlider(min=0.0001, max=1.0, step=0.0001, value=0.01, description="Alpha/L2", continuous_update=False)
reg_n_estimators_slider = IntSlider(min=10, max=500, step=10, value=100, description="N estimators", continuous_update=False)
reg_hidden_units_slider = IntSlider(min=5, max=200, value=50, description="Hidden units", continuous_update=False)
reg_mlp_alpha_slider    = FloatSlider(min=1e-5, max=0.1, value=0.0001, step=1e-4, description="MLP alpha", continuous_update=False)
reg_learning_rate_slider = FloatSlider(min=1e-4, max=0.1, value=0.001, step=1e-4, description="LR init", continuous_update=False)

advanced_box_reg = VBox([reg_test_size_slider, reg_random_state_slider])
advanced_acc_reg = Accordion(children=[advanced_box_reg])
advanced_acc_reg.set_title(0, "Advanced (test size / seed)")

reset_reg_btn = Button(description="Reset regression", button_style="warning", layout=Layout(width="95%"))
update_regression_btn = Button(description="Update regression plot", button_style="primary", layout=Layout(width="95%"))

def reset_regression(_):
    reg_model_selector.value = "Linear Regression"
    reg_dataset_selector.value = "linear"
    reg_colormap_selector.value = "viridis"
    reg_n_samples_slider.value = 300
    reg_noise_slider.value = 10
    reg_outliers_slider.value = 0
    reg_test_size_slider.value = 0.3
    reg_random_state_slider.value = 42
    reg_azimuth_slider.value = 45
    reg_elevation_slider.value = 30
    reg_alpha_slider.value = 0.6
    reg_point_size_slider.value = 20
    reg_alpha_reg_slider.value = 0.01
    reg_n_estimators_slider.value = 100
    reg_hidden_units_slider.value = 50
    reg_mlp_alpha_slider.value = 0.0001
    reg_learning_rate_slider.value = 0.001

reset_reg_btn.on_click(reset_regression)

regression_controls = VBox([
    Label("Regression"),
    reg_model_selector,
    reg_dataset_selector,
    reg_colormap_selector,
    reg_n_samples_slider,
    reg_noise_slider,
    reg_outliers_slider,
    advanced_acc_reg,
    reg_azimuth_slider,
    reg_elevation_slider,
    reg_alpha_slider,
    reg_point_size_slider,
    reg_alpha_reg_slider,
    reg_n_estimators_slider,
    reg_hidden_units_slider,
    reg_mlp_alpha_slider,
    reg_learning_rate_slider,
    update_regression_btn,
    reset_reg_btn
])

out_linear = Output()
out_nonlinear = Output()
out_regression = Output()

last_snapshot = {
    "linear": None,
    "nonlinear": None,
    "regression": None
}

def update_linear_plot(_=None):
    with out_linear:
        clear_output(wait=True)
        fig = plot_linear_classification(
            model_name=model_selector_lin.value,
            dataset=dataset_selector_lin.value,
            noise_pct=noise_slider_lin.value,
            outliers_pct=outliers_slider_lin.value,
            n_samples=n_samples_slider_lin.value,
            test_size=test_size_slider_lin.value,
            random_state=random_state_slider_lin.value,
            azimuth=azimuth_slider_lin.value,
            elevation=elevation_slider_lin.value,
            alpha_points=alpha_slider_lin.value,
            point_size=point_size_slider_lin.value,
            C=C_slider_lin.value,
            colormap=colormap_selector_lin.value,
            boundary_mode=boundary_mode_selector_lin.value,
            w1_manual=w1_slider_lin.value,
            w2_manual=w2_slider_lin.value,
            b_manual=b_slider_lin.value,
            animate=animate_checkbox.value,
            anim_interval=anim_interval_slider.value,
            anim_frames=anim_frames_slider.value
        )
        last_snapshot["linear"] = fig

def update_nonlinear_plot(_=None):
    with out_nonlinear:
        clear_output(wait=True)
        fig = plot_nonlinear_classification(
            model_name=model_selector_nl.value,
            dataset=dataset_selector_nl.value,
            poly_degree=poly_degree_slider_nl.value,
            noise_pct=noise_slider_nl.value,
            outliers_pct=outliers_slider_nl.value,
            n_samples=n_samples_slider_nl.value,
            test_size=test_size_slider_nl.value,
            random_state=random_state_slider_nl.value,
            azimuth=azimuth_slider_nl.value,
            elevation=elevation_slider_nl.value,
            alpha_points=alpha_slider_nl.value,
            point_size=point_size_slider_nl.value,
            colormap=colormap_selector_nl.value,
            C=C_slider_nl.value,
            gamma=gamma_slider_nl.value,
            svm_degree=svm_degree_slider.value,
            n_neighbors=n_neighbors_slider.value,
            weights=weights_knn_dropdown.value,
            max_depth=max_depth_slider.value,
            min_samples_split=min_samples_split_slider.value,
            n_estimators=n_estimators_slider.value,
            hidden_layer_sizes=hidden_units_slider.value,
            mlp_alpha=mlp_alpha_slider.value,
            learning_rate_init=learning_rate_slider.value,
            animate=animate_checkbox.value,
            anim_interval=anim_interval_slider.value,
            anim_frames=anim_frames_slider.value
        )
        last_snapshot["nonlinear"] = fig

def update_regression_plot(_=None):
    with out_regression:
        clear_output(wait=True)
        fig = plot_regression(
            model_name=reg_model_selector.value,
            dataset=reg_dataset_selector.value,
            noise_pct=reg_noise_slider.value,
            outliers_pct=reg_outliers_slider.value,
            n_samples=reg_n_samples_slider.value,
            test_size=reg_test_size_slider.value,
            random_state=reg_random_state_slider.value,
            azimuth=reg_azimuth_slider.value,
            elevation=reg_elevation_slider.value,
            alpha_points=reg_alpha_slider.value,
            point_size=reg_point_size_slider.value,
            colormap=reg_colormap_selector.value,
            alpha_reg=reg_alpha_reg_slider.value,
            n_estimators=reg_n_estimators_slider.value,
            hidden_units=reg_hidden_units_slider.value,
            mlp_alpha=reg_mlp_alpha_slider.value,
            learning_rate_init=reg_learning_rate_slider.value,
            animate=animate_checkbox.value,
            anim_interval=anim_interval_slider.value,
            anim_frames=anim_frames_slider.value
        )
        last_snapshot["regression"] = fig

update_linear_btn.on_click(update_linear_plot)
update_nonlinear_btn.on_click(update_nonlinear_plot)
update_regression_btn.on_click(update_regression_plot)

snapshot_btn = Button(
    description="Save snapshot",
    button_style="success",
    icon="camera",
    layout=Layout(width="95%")
)

snapshot_out = Output(layout=Layout(border="1px solid #444", width="95%", height="80px"))

def on_snapshot_click(b):
    snapshot_out.clear_output()
    idx = plots_tab.selected_index
    key = {0: "linear", 1: "nonlinear", 2: "regression"}.get(idx, "linear")
    fig = last_snapshot.get(key)

    with snapshot_out:
        if fig is None:
            print("No figure to save yet. Click 'Update' to draw a plot first.")
            return

        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"snapshot_{key}_{ts}.png"

        try:
            fig.savefig(fname, dpi=150, bbox_inches="tight")
            print(f"Snapshot saved as: {fname}")

            if IN_COLAB:
                files.download(fname)
            else:
                display(FileLink(fname))
                print("Click the link above to download the file to your device.")

        except Exception as e:
            print(f"Error saving figure:\n{e}")

snapshot_btn.on_click(on_snapshot_click)

data_section = VBox([
    Label("Data settings"),
    data_source_selector,
    upload_widget,
    csv_accordion,
    use_pca_checkbox
])

anim_section = VBox([
    Label("Animation / View"),
    animate_checkbox,
    anim_interval_slider,
    anim_frames_slider
])

controls_panel = VBox([
    data_section,
    anim_section,
    snapshot_btn,
    snapshot_out,
    linear_controls,
    nonlinear_controls,
    regression_controls
], layout=Layout(width="23%", padding="8px", border="1px solid #444"))

plots_tab = Tab(children=[out_linear, out_nonlinear, out_regression])
plots_tab.set_title(0, "Linear models")
plots_tab.set_title(1, "Nonlinear models")
plots_tab.set_title(2, "Regression")
plots_tab.layout = Layout(width="77%")

def on_tab_change(change):
    idx = change["new"]
    if idx == 0:
        linear_controls.layout.display = "block"
        nonlinear_controls.layout.display = "none"
        regression_controls.layout.display = "none"
        update_linear_plot()
    elif idx == 1:
        linear_controls.layout.display = "none"
        nonlinear_controls.layout.display = "block"
        regression_controls.layout.display = "none"
        update_nonlinear_plot()
    else:
        linear_controls.layout.display = "none"
        nonlinear_controls.layout.display = "none"
        regression_controls.layout.display = "block"
        update_regression_plot()

plots_tab.observe(on_tab_change, names="selected_index")

linear_controls.layout.display = "block"
nonlinear_controls.layout.display = "none"
regression_controls.layout.display = "none"

ui = HBox([controls_panel, plots_tab], layout=Layout(width="100%"))
display(ui)

update_linear_plot()

In [ ]:
!pip install -q --upgrade gradio

import gradio as gr
print("Gradio version:", gr.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 2.2 MB/s eta 0:00:00
Gradio version: 6.4.0


In [ ]:
import numpy as np
import pandas as pd
import io
import os
import tempfile
import base64
import matplotlib.pyplot as plt
from matplotlib import animation, rc
rc("animation", html="jshtml")
import matplotlib.gridspec as gridspec

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.pipeline import make_pipeline

from sklearn.metrics import roc_curve, auc, mean_squared_error, r2_score, confusion_matrix
from sklearn.base import BaseEstimator, ClassifierMixin

from sklearn.svm import SVC, LinearSVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, RandomForestRegressor,
    GradientBoostingClassifier, ExtraTreesClassifier
)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import (
    Perceptron, LogisticRegression, LinearRegression,
    Lasso, ElasticNet, Ridge, SGDClassifier
)

import gradio as gr
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# Core ML utilities
# ------------------------------------------------------------------

data_cache = {}

class RegressionClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_regressor):
        self.base_regressor = base_regressor

    def fit(self, X, y):
        self.base_regressor.fit(X, y)
        return self

    def decision_function(self, X):
        return self.base_regressor.predict(X)

    def predict(self, X):
        scores = self.decision_function(X)
        return (scores >= 0.5).astype(int)


def compute_roc_auc(clf, X_test, y_test):
    scores = None
    if hasattr(clf, "decision_function"):
        try:
            scores = clf.decision_function(X_test)
        except Exception:
            scores = None
    if scores is None and hasattr(clf, "predict_proba"):
        try:
            scores = clf.predict_proba(X_test)[:, 1]
        except Exception:
            scores = None
    if scores is None:
        try:
            scores = clf.predict(X_test)
        except Exception:
            scores = np.zeros(len(X_test))

    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_auc = auc(fpr, tpr)
    return fpr, tpr, roc_auc


def draw_summary_box(ax, lines):
    text = "\n".join(lines)
    ax.set_axis_off()
    ax.text(
        0.01,
        0.5,
        text,
        transform=ax.transAxes,
        va="center",
        ha="left",
        fontsize=9,
        family="monospace",
        bbox=dict(
            boxstyle="round",
            facecolor="#111827",
            edgecolor="#4b5563",
            alpha=0.95,
        ),
    )

def plot_learning_curve_ax(ax, estimator, X, y, task_type, random_state):
    try:
        scoring = "accuracy" if task_type == "cls" else "r2"
        train_sizes = np.linspace(0.2, 1.0, 5)

        train_sizes_abs, train_scores, val_scores = learning_curve(
            estimator,
            X,
            y,
            train_sizes=train_sizes,
            cv=3,
            scoring=scoring,
            shuffle=True,
            random_state=random_state,
        )

        train_mean = train_scores.mean(axis=1)
        val_mean = val_scores.mean(axis=1)

        ax.plot(train_sizes_abs, train_mean, "o-", label="Train")
        ax.plot(train_sizes_abs, val_mean, "o-", label="Validation")
        ax.set_xlabel("Training examples")
        ax.set_ylabel(scoring)
        ax.set_title("Learning curve", fontsize=10)
        ax.grid(alpha=0.3)
        ax.legend()
    except Exception as e:
        ax.text(
            0.5,
            0.5,
            f"LC error:\n{e}",
            ha="center",
            va="center",
            fontsize=8,
        )
        ax.set_axis_off()


def get_model_family(model):
    if isinstance(model, (LogisticRegression, LinearSVC, Perceptron, LinearDiscriminantAnalysis)):
        return "linear"
    if isinstance(model, (RandomForestClassifier, AdaBoostClassifier, DecisionTreeClassifier, ExtraTreesClassifier, GradientBoostingClassifier)):
        return "tree_ensemble"
    if isinstance(model, KNeighborsClassifier):
        return "knn"
    if isinstance(model, SVC):
        return "svm"
    if isinstance(model, MLPClassifier):
        return "mlp"
    return "other"


def get_reg_model_family(model):
    if isinstance(model, LinearRegression):
        return "linear"
    if isinstance(model, (RandomForestRegressor, DecisionTreeRegressor)):
        return "tree_ensemble"
    if isinstance(model, KNeighborsRegressor):
        return "knn"
    if isinstance(model, MLPRegressor):
        return "mlp"
    return "other"


# ---------------- Synthetic data generators ----------------

def make_linear_blobs(n_samples=300, noise=0.1, random_state=42):
    X, y = make_blobs(
        n_samples=n_samples,
        centers=[(-2, -2), (2, 2)],
        cluster_std=noise + 0.3,
        random_state=random_state,
    )
    return X, y


def make_linear_shifted(n_samples=300, noise=0.1, random_state=42):
    rng = np.random.RandomState(random_state)
    X1 = rng.randn(n_samples // 2, 2) * (0.3 + noise) + np.array([-2, 0])
    X2 = rng.randn(n_samples // 2, 2) * (0.3 + noise) + np.array([2, 0])
    X = np.vstack([X1, X2])
    y = np.hstack([np.zeros(len(X1)), np.ones(len(X2))])
    return X, y


def make_linear_vertical(n_samples=300, noise=0.1, random_state=42):
    rng = np.random.RandomState(random_state)
    X1 = rng.randn(n_samples // 2, 2) * (0.3 + noise) + np.array([0, -2])
    X2 = rng.randn(n_samples // 2, 2) * (0.3 + noise) + np.array([0, 2])
    X = np.vstack([X1, X2])
    y = np.hstack([np.zeros(len(X1)), np.ones(len(X2))])
    return X, y


def make_spiral(n_samples=300, noise=0.2, random_state=None):
    rng = np.random.RandomState(random_state)
    n = n_samples // 2
    theta = np.sqrt(rng.rand(n)) * 2 * np.pi
    r_a = 2 * theta + np.pi
    data_a = np.c_[r_a * np.cos(theta), r_a * np.sin(theta)]
    theta = np.sqrt(rng.rand(n)) * 2 * np.pi
    r_b = -2 * theta - np.pi
    data_b = np.c_[r_b * np.cos(theta), r_b * np.sin(theta)]
    X = np.vstack([data_a, data_b])
    X += rng.normal(scale=noise, size=X.shape)
    y = np.hstack([np.zeros(n, dtype=int), np.ones(n, dtype=int)])
    return X, y


def make_xor(n_samples=300, noise=0.2, random_state=None):
    rng = np.random.RandomState(random_state)
    X = rng.randn(n_samples, 2)
    y = (X[:, 0] * X[:, 1] > 0).astype(int)
    X += rng.normal(scale=noise, size=X.shape)
    return X, y


def make_checkerboard(n_samples=300, noise=0.1, random_state=None):
    rng = np.random.RandomState(random_state)
    X = rng.uniform(-2, 2, size=(n_samples, 2))
    y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(int)
    X += rng.normal(scale=noise * 0.2, size=X.shape)
    return X, y


def generate_linear_data(kind, n_samples, noise_std, random_state):
    if kind == "blobs":
        return make_linear_blobs(n_samples, noise_std, random_state)
    if kind == "shifted":
        return make_linear_shifted(n_samples, noise_std, random_state)
    if kind == "vertical":
        return make_linear_vertical(n_samples, noise_std, random_state)
    return make_linear_blobs(n_samples, noise_std, random_state)


def generate_nonlinear_data(kind, n_samples, noise_std, random_state):
    if kind == "moons":
        return make_moons(n_samples=n_samples, noise=noise_std, random_state=random_state)
    if kind == "circles":
        return make_circles(n_samples=n_samples, noise=noise_std, factor=0.5, random_state=random_state)
    if kind == "blobs":
        X, y = make_blobs(
            n_samples=n_samples,
            centers=[(-1, -1), (1, 1)],
            cluster_std=0.8,
            random_state=random_state,
        )
        return X, y
    if kind == "xor":
        return make_xor(n_samples, noise_std, random_state)
    if kind == "spiral":
        return make_spiral(n_samples, noise_std, random_state)
    if kind == "checkerboard":
        return make_checkerboard(n_samples, noise_std, random_state)
    return make_moons(n_samples=n_samples, noise=noise_std, random_state=random_state)


def generate_regression_data(kind, n_samples, noise_std, random_state):
    rng = np.random.RandomState(random_state)
    X = rng.uniform(-3, 3, size=(n_samples, 2))
    if kind == "linear":
        w = np.array([2.0, -3.0])
        y = X @ w + 10
    elif kind == "quadratic":
        y = 2 * X[:, 0] ** 2 - 3 * X[:, 1] + 5
    else:
        y = X[:, 0] * X[:, 1] + 0.5 * X[:, 0] ** 2
    y = y + rng.normal(scale=noise_std, size=y.shape)
    return X, y


def apply_noise_outliers_classification(X, y, noise_pct, outliers_pct, random_state):
    rng = np.random.RandomState(random_state + 123)
    X = X.copy()
    y = y.copy()
    n = len(y)

    num_noise = int(noise_pct / 100.0 * n)
    if num_noise > 0:
        idx = rng.choice(n, size=num_noise, replace=False)
        y[idx] = 1 - y[idx]

    num_out = int(outliers_pct / 100.0 * n)
    if num_out > 0:
        idx = rng.choice(n, size=num_out, replace=False)
        X[idx] = rng.uniform(-6, 6, size=(num_out, X.shape[1]))
        y[idx] = rng.randint(0, 2, size=num_out)

    return X, y


def apply_noise_outliers_regression(X, y, noise_pct, outliers_pct, random_state):
    rng = np.random.RandomState(random_state + 456)
    X = X.copy()
    y = y.copy()
    n = len(y)

    num_noise = int(noise_pct / 100.0 * n)
    if num_noise > 0:
        idx = rng.choice(n, size=num_noise, replace=False)
        y_std = y.std() if y.std() > 0 else 1.0
        y[idx] += rng.normal(scale=0.5 * y_std, size=num_noise)

    num_out = int(outliers_pct / 100.0 * n)
    if num_out > 0:
        idx = rng.choice(n, size=num_out, replace=False)
        y_std = y.std() if y.std() > 0 else 1.0
        y[idx] += rng.normal(loc=5 * y_std, scale=2 * y_std, size=num_out)

    return X, y


# ---------------- Model builders ----------------

linear_model_names = [
    "Perceptron",
    "Linear SVM",
    "Logistic Regression",
    "LDA",
]


def build_linear_model(name, C=1.0):
    if name == "Linear SVM":
        return LinearSVC(C=C, max_iter=5000)
    if name == "Logistic Regression":
        return LogisticRegression(C=C, max_iter=2000)
    if name == "Perceptron":
        return Perceptron(max_iter=2000, tol=1e-3)
    if name == "LDA":
        return LinearDiscriminantAnalysis()
    return LogisticRegression(C=C, max_iter=2000)


def build_nonlinear_model(
    name,
    C=1.0,
    gamma=1.0,
    svm_degree=3,
    n_neighbors=7,
    weights="uniform",
    max_depth=6,
    min_samples_split=2,
    n_estimators=100,
    hidden_layer_sizes=50,
    mlp_alpha=0.0001,
    learning_rate_init=0.001,
):
    if name == "SVM (RBF)":
        return SVC(kernel="rbf", probability=True, C=C, gamma=gamma)
    if name == "SVM (Poly)":
        return SVC(kernel="poly", probability=True, C=C, gamma=gamma, degree=svm_degree)
    if name == "Linear SVM":
        return SVC(kernel="linear", probability=True, C=C)
    if name == "KNN":
        return KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights)
    if name == "Decision Tree":
        return DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split)
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)
    if name == "AdaBoost":
        return AdaBoostClassifier(n_estimators=n_estimators)
    if name == "MLP Neural Network":
        return MLPClassifier(
            hidden_layer_sizes=(hidden_layer_sizes,),
            alpha=mlp_alpha,
            learning_rate_init=learning_rate_init,
            max_iter=1000,
        )
    if name == "Naive Bayes":
        return GaussianNB()
    if name == "LDA":
        return LinearDiscriminantAnalysis()
    if name == "QDA":
        return QuadraticDiscriminantAnalysis()
    if name == "Perceptron":
        return Perceptron(max_iter=1000, tol=1e-3)
    if name == "Logistic Regression":
        return LogisticRegression(C=C, max_iter=2000)
    if name == "SGD (logistic)":
        return SGDClassifier(loss="log_loss", max_iter=2000, alpha=C)
    if name == "Gradient Boosting":
        return GradientBoostingClassifier()
    if name == "Extra Trees":
        return ExtraTreesClassifier(n_estimators=n_estimators, max_depth=max_depth)
    return SVC(kernel="rbf", probability=True, C=C, gamma=gamma)


def build_regression_model(
    name,
    alpha=1.0,
    n_estimators=100,
    hidden_units=50,
    mlp_alpha=0.0001,
    learning_rate_init=0.001,
):
    if name == "Linear Regression":
        return LinearRegression()
    if name == "Random Forest Regressor":
        return RandomForestRegressor(n_estimators=n_estimators, max_depth=6)
    if name == "Decision Tree Regressor":
        return DecisionTreeRegressor(max_depth=6)
    if name == "KNN Regressor":
        return KNeighborsRegressor(n_neighbors=7, weights="distance")
    if name == "MLP Regressor":
        return MLPRegressor(
            hidden_layer_sizes=(hidden_units,),
            alpha=mlp_alpha,
            learning_rate_init=learning_rate_init,
            max_iter=1000,
        )
    if name == "Ridge Regression":
        return Ridge(alpha=alpha)
    if name == "Lasso Regression":
        return Lasso(alpha=alpha, max_iter=5000)
    if name == "ElasticNet Regression":
        return ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=5000)
    if name == "SVR (RBF)":
        return SVR(C=1.0, gamma="scale")
    return LinearRegression()


# ---------------- Data access for synthetic / CSV ----------------

def get_data_for_task(
    task,
    dataset_name,
    n_samples,
    noise_pct,
    outliers_pct,
    random_state,
    data_source,
    df,
    target_col,
    feature_col,
):
    # CSV branch
    if data_source == "Uploaded CSV" and df is not None:
        if target_col is None or feature_col is None:
            raise ValueError("Please choose target & feature columns.")
        if feature_col not in df.columns or target_col not in df.columns:
            raise ValueError("Selected columns not found in CSV.")
        X = df[[feature_col]].values
        if X.shape[1] > 1:
            X = X[:, :1]

        rng = np.random.RandomState(random_state)
        X2 = rng.normal(scale=1.0, size=(X.shape[0], 1))
        X = np.hstack([X, X2])

        y = df[target_col].values

        if task in ["linear_cls", "nonlinear_cls"]:
            if len(np.unique(y)) > 2:
                median = np.median(y)
                y = (y > median).astype(int)
            else:
                uniques = np.unique(y)
                if set(uniques) != {0, 1}:
                    mapping = {uniques[0]: 0, uniques[1]: 1}
                    y = np.vectorize(mapping.get)(y)
            X, y = apply_noise_outliers_classification(
                X,
                y,
                noise_pct=noise_pct,
                outliers_pct=outliers_pct,
                random_state=random_state,
            )
        else:
            X, y = apply_noise_outliers_regression(
                X,
                y,
                noise_pct=noise_pct,
                outliers_pct=outliers_pct,
                random_state=random_state,
            )
        return X, y

    # synthetic branch + caching
    key = (task, dataset_name, n_samples, noise_pct, outliers_pct, random_state)
    if key in data_cache:
        return data_cache[key]

    noise_std = 0.2
    if task == "regression":
        noise_std = 0.01 + (noise_pct / 100.0) * 1.0

    if task == "linear_cls":
        X, y = generate_linear_data(dataset_name, n_samples, noise_std, random_state)
        X, y = apply_noise_outliers_classification(
            X,
            y,
            noise_pct=noise_pct,
            outliers_pct=outliers_pct,
            random_state=random_state,
        )
    elif task == "nonlinear_cls":
        X, y = generate_nonlinear_data(dataset_name, n_samples, noise_std, random_state)
        X, y = apply_noise_outliers_classification(
            X,
            y,
            noise_pct=noise_pct,
            outliers_pct=outliers_pct,
            random_state=random_state,
        )
    elif task == "regression":
        X, y = generate_regression_data(dataset_name, n_samples, noise_std, random_state)
        X, y = apply_noise_outliers_regression(
            X,
            y,
            noise_pct=noise_pct,
            outliers_pct=outliers_pct,
            random_state=random_state,
        )
    else:
        raise ValueError("Unknown task")

    data_cache[key] = (X, y)
    return X, y


def maybe_pca_transform(X, use_pca):
    if not use_pca:
        return X
    if X.shape[1] <= 2:
        return X
    pca = PCA(n_components=2)
    return pca.fit_transform(X)



# ------------------------------------------------------------------
# 3D rotation helper – returns HTML + GIF path (surface + points)
# ------------------------------------------------------------------
from matplotlib import animation
def make_3d_rotation_html_and_gif(
    xx,
    yy,
    surface_vals,
    X_vis,
    colors,
    colormap,
    elevation,
    frames,
    interval,
    title="3D rotating view",
):
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection="3d")

    # 3D surface
    surf = ax.plot_surface(
        xx,
        yy,
        surface_vals,
        rstride=1,
        cstride=1,
        alpha=0.75,
        cmap=colormap,
        edgecolor="k",
    )

    # نقاط البيانات تظهر في الـ GIF
    if X_vis is not None and colors is not None:
        ax.scatter(
            X_vis[:, 0],
            X_vis[:, 1],
            colors,
            c=colors,
            cmap=colormap,
            s=20,
            edgecolors="k",
            alpha=0.7,
        )

    ax.set_xlabel("X1")
    ax.set_ylabel("X2")
    ax.set_zlabel("Score / Pred")
    ax.set_title(title, fontsize=10)

    def update(angle):
        ax.view_init(elev=elevation, azim=angle)
        return (surf,)

    frame_vals = np.linspace(0, 360, int(frames))
    ani = animation.FuncAnimation(
        fig, update, frames=frame_vals, interval=interval, blit=False
    )

    # نحفظ GIF في ملف مؤقت
    tmpdir = tempfile.mkdtemp()
    gif_path = os.path.join(tmpdir, "rotation.gif")
    ani.save(gif_path, writer="pillow", dpi=120)

    # نقرأه كـ base64 ونبنيه كصورة داخل HTML عشان يشتغل داخل Gradio
    with open(gif_path, "rb") as f:
        gif_bytes = f.read()
    b64 = base64.b64encode(gif_bytes).decode("ascii")

    html_anim = f"""
    <div style="text-align:center;">
        <img src="data:image/gif;base64,{b64}"
             style="max-width:100%; height:auto; border-radius:8px;" />
    </div>
    """

    plt.close(fig)
    return html_anim, gif_path


# ------------------------------------------------------------------
# Core experiment functions (return fig + metrics + GIF + file paths)
# ------------------------------------------------------------------

def linear_experiment_core(
    data_source,
    df,
    use_pca,
    target_col,
    feature_col,
    model_name,
    dataset,
    noise_pct,
    outliers_pct,
    n_samples,
    test_size,
    random_state,
    azimuth,
    elevation,
    alpha_points,
    point_size,
    C,
    colormap,
    boundary_mode,
    w1_manual,
    w2_manual,
    b_manual,
    animate,
    anim_interval,
    anim_frames,
):
    try:
        X, y = get_data_for_task(
            task="linear_cls",
            dataset_name=dataset,
            n_samples=n_samples,
            noise_pct=noise_pct,
            outliers_pct=outliers_pct,
            random_state=random_state,
            data_source=data_source,
            df=df,
            target_col=target_col,
            feature_col=feature_col,
        )
    except Exception as e:
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"Data error:\n{e}", ha="center", va="center")
        ax.set_axis_off()
        return fig, "", "", None, None

    X_vis = maybe_pca_transform(X, use_pca)

    X_train, X_test, y_train, y_test = train_test_split(
        X_vis, y, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler = StandardScaler()
    model = build_linear_model(model_name, C=C)
    pipeline = make_pipeline(scaler, model)
    base_model = model

    scores_test = None

    if boundary_mode == "Model":
        pipeline.fit(X_train, y_train)
        y_train_pred = pipeline.predict(X_train)
        y_test_pred = pipeline.predict(X_test)
        train_acc = np.mean(y_train_pred == y_train)

        fpr, tpr, roc_auc = compute_roc_auc(pipeline, X_test, y_test)

        if hasattr(pipeline, "decision_function"):
            try:
                scores_test = pipeline.decision_function(X_test)
            except Exception:
                try:
                    scores_test = pipeline.predict_proba(X_test)[:, 1]
                except Exception:
                    scores_test = pipeline.predict(X_test).astype(float)
        elif hasattr(pipeline, "predict_proba"):
            scores_test = pipeline.predict_proba(X_test)[:, 1]
        else:
            scores_test = pipeline.predict(X_test).astype(float)

    else:
        scores_train = (
            w1_manual * X_train[:, 0] + w2_manual * X_train[:, 1] + b_manual
        )
        y_train_pred = (scores_train > 0).astype(int)
        scores_test = w1_manual * X_test[:, 0] + w2_manual * X_test[:, 1] + b_manual
        y_test_pred = (scores_test > 0).astype(int)
        train_acc = np.mean(y_train_pred == y_train)
        fpr, tpr, _ = roc_curve(y_test, scores_test)
        roc_auc = auc(fpr, tpr)
        pipeline = None
        base_model = None

    cm = confusion_matrix(y_test, y_test_pred)

    x_min, x_max = X_vis[:, 0].min() - 0.6, X_vis[:, 0].max() + 0.6
    y_min, y_max = X_vis[:, 1].min() - 0.6, X_vis[:, 1].max() + 0.6
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 100),
        np.linspace(y_min, y_max, 100),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    if boundary_mode == "Model":
        try:
            Z = pipeline.predict(grid).reshape(xx.shape)
        except Exception:
            Z = np.zeros(xx.shape)
        if hasattr(pipeline, "decision_function"):
            try:
                margin_vals = pipeline.decision_function(grid).reshape(xx.shape)
            except Exception:
                margin_vals = None
        else:
            margin_vals = None
        model_vals_surface = margin_vals if margin_vals is not None else Z.astype(float)
        manual_vals = w1_manual * xx + w2_manual * yy + b_manual
    else:
        scores_grid = w1_manual * grid[:, 0] + w2_manual * grid[:, 1] + b_manual
        Z = (scores_grid > 0).astype(int).reshape(xx.shape)
        margin_vals = w1_manual * xx + w2_manual * yy + b_manual
        model_vals_surface = margin_vals
        manual_vals = None

    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(3, 3, height_ratios=[0.6, 2, 2], figure=fig)

    ax_sum = fig.add_subplot(gs[0, :])
    summary_lines = [
        f"Data source : {data_source}   |   Dataset: {dataset}",
        f"Model      : {model_name}   |   samples: {len(X)} (features: {X.shape[1]})",
        f"Noise %    : {noise_pct:.1f}   |   Outliers %: {outliers_pct:.1f}   |   "
        f"Test size: {test_size:.2f}   |   Rand seed: {random_state}",
        f"Train acc  : {train_acc:.3f}   |   AUC: {roc_auc:.3f}   |   Mode: {boundary_mode}",
    ]
    draw_summary_box(ax_sum, summary_lines)

    ax1 = fig.add_subplot(gs[1, 0])
    ax1.contourf(
        xx,
        yy,
        Z,
        alpha=0.35,
        cmap=colormap,
        levels=np.linspace(0, 1, 3),
    )
    ax1.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y,
        s=point_size,
        cmap=colormap,
        edgecolors="k",
        alpha=alpha_points,
    )
    ax1.set_title("Decision regions", fontsize=10)

    if boundary_mode == "Model":
        if margin_vals is not None:
            ax1.contour(
                xx,
                yy,
                margin_vals,
                levels=[-1, 1],
                colors="black",
                linestyles="--",
                linewidths=2,
            )
        if manual_vals is not None:
            ax1.contour(
                xx,
                yy,
                manual_vals,
                levels=[0],
                colors="yellow",
                linestyles="--",
                linewidths=2,
            )
    else:
        if margin_vals is not None:
            ax1.contour(
                xx,
                yy,
                margin_vals,
                levels=[0],
                colors="yellow",
                linestyles="--",
                linewidths=2,
            )

    ax2 = fig.add_subplot(gs[1, 1], projection="3d")
    ax2.plot_surface(
        xx,
        yy,
        model_vals_surface,
        rstride=1,
        cstride=1,
        alpha=alpha_points,
        cmap=colormap,
        edgecolor="k",
    )
    ax2.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        y,
        c=y,
        s=point_size,
        cmap=colormap,
        edgecolors="k",
        alpha=alpha_points,
    )
    ax2.set_xlabel("X1")
    ax2.set_ylabel("X2")
    ax2.set_zlabel("Score / Pred")
    ax2.set_title("3D decision surface", fontsize=10)
    ax2.view_init(elev=elevation, azim=azimuth)

    ax3 = fig.add_subplot(gs[1, 2])
    ax3.plot(fpr, tpr, lw=3, label=f"AUC = {roc_auc:.3f}")
    ax3.plot([0, 1], [0, 1], "--", color="gray", label="random")
    ax3.set_xlabel("FPR")
    ax3.set_ylabel("TPR")
    ax3.set_title("ROC Curve (test set)", fontsize=10)
    ax3.legend()
    ax3.grid(alpha=0.25)

    ax4 = fig.add_subplot(gs[2, 0])
    ax4.imshow(cm, interpolation="nearest", cmap="Blues")
    ax4.set_title("Confusion matrix", fontsize=10)
    ax4.set_xlabel("Predicted label")
    ax4.set_ylabel("True label")
    for (i, j), v in np.ndenumerate(cm):
        ax4.text(j, i, str(v), ha="center", va="center", color="black", fontsize=8)

    ax5 = fig.add_subplot(gs[2, 1])
    if pipeline is not None:
        plot_learning_curve_ax(
            ax5,
            pipeline,
            X_vis,
            y,
            task_type="cls",
            random_state=random_state,
        )
    else:
        ax5.text(
            0.5,
            0.5,
            "Learning curve\nnot available in Manual mode",
            ha="center",
            va="center",
            fontsize=9,
        )
        ax5.set_axis_off()

    ax6 = fig.add_subplot(gs[2, 2])
    if pipeline is not None and base_model is not None:
        family = get_model_family(base_model)
        if family == "linear" and hasattr(base_model, "coef_"):
            coefs = base_model.coef_.ravel()
            ax6.bar(range(len(coefs)), coefs)
            ax6.set_title("Coefficients", fontsize=10)
            ax6.set_xlabel("Feature index")
            ax6.set_ylabel("Weight")
        elif hasattr(base_model, "feature_importances_"):
            imps = base_model.feature_importances_
            ax6.bar(range(len(imps)), imps)
            ax6.set_title("Feature importances", fontsize=10)
            ax6.set_xlabel("Feature index")
            ax6.set_ylabel("Importance")
        else:
            ax6.hist(scores_test[y_test == 0], bins=15, alpha=0.7, label="class 0")
            ax6.hist(scores_test[y_test == 1], bins=15, alpha=0.7, label="class 1")
            ax6.set_title("Score distribution (test)", fontsize=10)
            ax6.set_xlabel("Score")
            ax6.set_ylabel("Count")
            ax6.legend()
    else:
        ax6.hist(scores_test[y_test == 0], bins=15, alpha=0.7, label="class 0")
        ax6.hist(scores_test[y_test == 1], bins=15, alpha=0.7, label="class 1")
        ax6.set_title("Score distribution (test)", fontsize=10)
        ax6.set_xlabel("Score")
        ax6.set_ylabel("Count")
        ax6.legend()

    plt.tight_layout()

    metrics_md = "### Metrics (linear model)\n" + "\n".join(
      [f"- {line}" for line in summary_lines[1:]]
    )

    # Always save PNG of the full visualization
    tmpdir_png = tempfile.mkdtemp()
    png_path = os.path.join(tmpdir_png, "linear_visualization.png")
    fig.savefig(png_path, dpi=140, bbox_inches="tight")

    gif_html = ""
    gif_path = None

    if animate:
        gif_html, gif_path = make_3d_rotation_html_and_gif(
          xx,
          yy,
          model_vals_surface,
          X_vis,
          y,
          colormap,
          elevation,
          anim_frames,
          anim_interval,
          title="3D surface (linear)",
        )


    return fig, metrics_md, gif_html, png_path, gif_path


def nonlinear_experiment_core(
    data_source,
    df,
    use_pca,
    target_col,
    feature_col,
    model_name,
    dataset,
    poly_degree,
    noise_pct,
    outliers_pct,
    n_samples,
    test_size,
    random_state,
    azimuth,
    elevation,
    alpha_points,
    point_size,
    colormap,
    C,
    gamma,
    svm_degree,
    n_neighbors,
    weights,
    max_depth,
    min_samples_split,
    n_estimators,
    hidden_layer_sizes,
    mlp_alpha,
    learning_rate_init,
    animate,
    anim_interval,
    anim_frames,
):
    try:
        X, y = get_data_for_task(
            task="nonlinear_cls",
            dataset_name=dataset,
            n_samples=n_samples,
            noise_pct=noise_pct,
            outliers_pct=outliers_pct,
            random_state=random_state,
            data_source=data_source,
            df=df,
            target_col=target_col,
            feature_col=feature_col,
        )
    except Exception as e:
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"Data error:\n{e}", ha="center", va="center")
        ax.set_axis_off()
        return fig, "", "", None, None

    X_vis = maybe_pca_transform(X, use_pca)

    X_train, X_test, y_train, y_test = train_test_split(
        X_vis, y, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler = StandardScaler()
    if poly_degree > 1:
        poly = PolynomialFeatures(degree=poly_degree, include_bias=False)
        base_model = build_nonlinear_model(
            model_name,
            C=C,
            gamma=gamma,
            svm_degree=svm_degree,
            n_neighbors=n_neighbors,
            weights=weights,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            n_estimators=n_estimators,
            hidden_layer_sizes=hidden_layer_sizes,
            mlp_alpha=mlp_alpha,
            learning_rate_init=learning_rate_init,
        )
        pipeline = make_pipeline(poly, scaler, base_model)
    else:
        base_model = build_nonlinear_model(
            model_name,
            C=C,
            gamma=gamma,
            svm_degree=svm_degree,
            n_neighbors=n_neighbors,
            weights=weights,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            n_estimators=n_estimators,
            hidden_layer_sizes=hidden_layer_sizes,
            mlp_alpha=mlp_alpha,
            learning_rate_init=learning_rate_init,
        )
        pipeline = make_pipeline(scaler, base_model)

    pipeline.fit(X_train, y_train)
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    train_acc = np.mean(y_train_pred == y_train)

    fpr, tpr, roc_auc = compute_roc_auc(pipeline, X_test, y_test)

    if hasattr(pipeline, "decision_function"):
        try:
            scores_test = pipeline.decision_function(X_test)
        except Exception:
            try:
                scores_test = pipeline.predict_proba(X_test)[:, 1]
            except Exception:
                scores_test = pipeline.predict(X_test).astype(float)
    elif hasattr(pipeline, "predict_proba"):
        scores_test = pipeline.predict_proba(X_test)[:, 1]
    else:
        scores_test = pipeline.predict(X_test).astype(float)

    cm = confusion_matrix(y_test, y_test_pred)

    x_min, x_max = X_vis[:, 0].min() - 0.6, X_vis[:, 0].max() + 0.6
    y_min, y_max = X_vis[:, 1].min() - 0.6, X_vis[:, 1].max() + 0.6
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 100),
        np.linspace(y_min, y_max, 100),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    try:
        Z = pipeline.predict(grid).reshape(xx.shape)
    except Exception:
        try:
            Z = pipeline.predict_proba(grid)[:, 1].reshape(xx.shape)
        except Exception:
            Z = np.zeros(xx.shape)

    margin_vals = None
    if model_name in ["SVM (RBF)", "SVM (Poly)", "Linear SVM"]:
        try:
            margin_vals = pipeline.decision_function(grid).reshape(xx.shape)
        except Exception:
            margin_vals = None

    family = get_model_family(base_model)

    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(3, 3, height_ratios=[0.6, 2, 2], figure=fig)

    ax_sum = fig.add_subplot(gs[0, :])
    summary_lines = [
        f"Data source : {data_source}   |   Dataset: {dataset}",
        f"Model      : {model_name}   |   samples: {len(X)} (features: {X.shape[1]})",
        f"Noise %    : {noise_pct:.1f}   |   Outliers %: {outliers_pct:.1f}   |   "
        f"Test size: {test_size:.2f}   |   Rand seed: {random_state}",
        f"Train acc  : {train_acc:.3f}   |   AUC: {roc_auc:.3f}",
    ]
    draw_summary_box(ax_sum, summary_lines)

    ax1 = fig.add_subplot(gs[1, 0])
    ax1.contourf(
        xx,
        yy,
        Z,
        alpha=0.35,
        cmap=colormap,
        levels=np.linspace(0, 1, 3),
    )
    ax1.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y,
        s=point_size,
        cmap=colormap,
        edgecolors="k",
        alpha=alpha_points,
    )
    ax1.set_title(f"{model_name} | {dataset} | degree={poly_degree}", fontsize=10)

    if family == "svm" and hasattr(base_model, "support_"):
        sv_idx = base_model.support_
        sv_points = X_train[sv_idx]
        ax1.scatter(
            sv_points[:, 0],
            sv_points[:, 1],
            s=point_size * 1.8,
            facecolors="none",
            edgecolors="yellow",
            linewidths=1.5,
            label="SV",
        )
        ax1.legend(fontsize=8)

    if margin_vals is not None:
        ax1.contour(
            xx,
            yy,
            margin_vals,
            levels=[-1, 1],
            colors="yellow",
            linestyles="--",
            linewidths=2,
        )

    ax2 = fig.add_subplot(gs[1, 1], projection="3d")
    ax2.plot_surface(
        xx,
        yy,
        Z.astype(float),
        rstride=1,
        cstride=1,
        alpha=alpha_points,
        cmap=colormap,
        edgecolor="k",
    )
    ax2.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        y,
        c=y,
        s=point_size,
        cmap=colormap,
        edgecolors="k",
        alpha=alpha_points,
    )
    ax2.set_xlabel("X1")
    ax2.set_ylabel("X2")
    ax2.set_zlabel("Prediction/Prob")
    ax2.set_title("3D decision surface", fontsize=10)
    ax2.view_init(elev=elevation, azim=azimuth)

    ax3 = fig.add_subplot(gs[1, 2])
    ax3.plot(fpr, tpr, lw=3, label=f"AUC = {roc_auc:.3f}")
    ax3.plot([0, 1], [0, 1], "--", color="gray", label="random")
    ax3.set_xlabel("False Positive Rate")
    ax3.set_ylabel("True Positive Rate")
    ax3.set_title("ROC Curve (test set)", fontsize=10)
    ax3.legend()
    ax3.grid(alpha=0.25)

    ax4 = fig.add_subplot(gs[2, 0])
    ax4.imshow(cm, interpolation="nearest", cmap="Blues")
    ax4.set_title("Confusion matrix", fontsize=10)
    ax4.set_xlabel("Predicted label")
    ax4.set_ylabel("True label")
    for (i, j), v in np.ndenumerate(cm):
        ax4.text(j, i, str(v), ha="center", va="center", color="black", fontsize=8)

    ax5 = fig.add_subplot(gs[2, 1])
    plot_learning_curve_ax(
        ax5,
        pipeline,
        X_vis,
        y,
        task_type="cls",
        random_state=random_state,
    )

    ax6 = fig.add_subplot(gs[2, 2])

    if family == "linear" and hasattr(base_model, "coef_"):
        coefs = base_model.coef_.ravel()
        ax6.bar(range(len(coefs)), coefs)
        ax6.set_title("Coefficients", fontsize=10)
        ax6.set_xlabel("Feature index")
        ax6.set_ylabel("Weight")
    elif family == "tree_ensemble" and hasattr(base_model, "feature_importances_"):
        imps = base_model.feature_importances_
        ax6.bar(range(len(imps)), imps)
        ax6.set_title("Feature importances", fontsize=10)
        ax6.set_xlabel("Feature index")
        ax6.set_ylabel("Importance")
    elif family == "knn":
        base_k = base_model.n_neighbors
        ks = list(range(max(1, base_k - 4), base_k + 5))
        errors = []
        for k in ks:
            knn_tmp = KNeighborsClassifier(n_neighbors=k, weights=base_model.weights)
            pipe_tmp = make_pipeline(StandardScaler(), knn_tmp)
            pipe_tmp.fit(X_train, y_train)
            y_val_pred = pipe_tmp.predict(X_test)
            err = 1 - np.mean(y_val_pred == y_test)
            errors.append(err)
        ax6.plot(ks, errors, "o-")
        ax6.set_title("Error vs K", fontsize=10)
        ax6.set_xlabel("K")
        ax6.set_ylabel("Error")
    elif family == "mlp" and hasattr(base_model, "loss_curve_"):
        losses = base_model.loss_curve_
        ax6.plot(range(1, len(losses) + 1), losses, "o-")
        ax6.set_title("MLP loss curve", fontsize=10)
        ax6.set_xlabel("Epoch")
        ax6.set_ylabel("Loss")
    else:
        ax6.hist(scores_test[y_test == 0], bins=15, alpha=0.7, label="class 0")
        ax6.hist(scores_test[y_test == 1], bins=15, alpha=0.7, label="class 1")
        ax6.set_title("Score distribution (test)", fontsize=10)
        ax6.set_xlabel("Score")
        ax6.set_ylabel("Count")
        ax6.legend()

    plt.tight_layout()

    metrics_md = "### Metrics (nonlinear model)\n" + "\n".join(
      [f"- {line}" for line in summary_lines[1:]]
    )

    # Always save PNG of the full visualization
    tmpdir_png = tempfile.mkdtemp()
    png_path = os.path.join(tmpdir_png, "nonlinear_visualization.png")
    fig.savefig(png_path, dpi=140, bbox_inches="tight")

    gif_html = ""
    gif_path = None

    if animate:
        gif_html, gif_path = make_3d_rotation_html_and_gif(
          xx,
          yy,
          Z.astype(float),
          X_vis,
          y,
          colormap,
          elevation,
          anim_frames,
          anim_interval,
          title="3D surface (nonlinear)",
        )

    return fig, metrics_md, gif_html, png_path, gif_path


def regression_experiment_core(
    data_source,
    df,
    use_pca,
    target_col,
    feature_col,
    model_name,
    dataset,
    noise_pct,
    outliers_pct,
    n_samples,
    test_size,
    random_state,
    azimuth,
    elevation,
    alpha_points,
    point_size,
    colormap,
    alpha_reg,
    n_estimators,
    hidden_units,
    mlp_alpha,
    learning_rate_init,
    animate,
    anim_interval,
    anim_frames,
):
    try:
        X, y = get_data_for_task(
            task="regression",
            dataset_name=dataset,
            n_samples=n_samples,
            noise_pct=noise_pct,
            outliers_pct=outliers_pct,
            random_state=random_state,
            data_source=data_source,
            df=df,
            target_col=target_col,
            feature_col=feature_col,
        )
    except Exception as e:
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"Data error:\n{e}", ha="center", va="center")
        ax.set_axis_off()
        return fig, "", "", None, None

    X_vis = maybe_pca_transform(X, use_pca)

    X_train, X_test, y_train, y_test = train_test_split(
        X_vis, y, test_size=test_size, random_state=random_state
    )

    scaler = StandardScaler()
    model = build_regression_model(
        model_name,
        alpha=alpha_reg,
        n_estimators=n_estimators,
        hidden_units=hidden_units,
        mlp_alpha=mlp_alpha,
        learning_rate_init=learning_rate_init,
    )
    pipeline = make_pipeline(scaler, model)
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    x_min, x_max = X_vis[:, 0].min() - 0.6, X_vis[:, 0].max() + 0.6
    y_min, y_max = X_vis[:, 1].min() - 0.6, X_vis[:, 1].max() + 0.6
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 50),
        np.linspace(y_min, y_max, 50),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    z_grid = pipeline.predict(grid).reshape(xx.shape)

    base_model = pipeline[-1]
    family = get_reg_model_family(base_model)

    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(3, 3, height_ratios=[0.6, 2, 2], figure=fig)

    ax_sum = fig.add_subplot(gs[0, :])
    summary_lines = [
        f"Data source : {data_source}   |   Dataset: {dataset}",
        f"Model      : {model_name}   |   samples: {len(X)} (features: {X.shape[1]})",
        f"Noise %    : {noise_pct:.1f}   |   Outliers %: {outliers_pct:.1f}   |   "
        f"Test size: {test_size:.2f}   |   Rand seed: {random_state}",
        f"MSE        : {mse:.3f}   |   R²: {r2:.3f}",
    ]
    draw_summary_box(ax_sum, summary_lines)

    ax1 = fig.add_subplot(gs[1, 0])
    sc = ax1.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        c=y,
        s=point_size,
        cmap=colormap,
        edgecolors="k",
        alpha=alpha_points,
    )
    ax1.set_title(f"{model_name} | {dataset}", fontsize=10)
    ax1.set_xlabel("X1")
    ax1.set_ylabel("X2")
    plt.colorbar(sc, ax=ax1, label="y")

    ax2 = fig.add_subplot(gs[1, 1], projection="3d")
    ax2.plot_surface(
        xx,
        yy,
        z_grid,
        rstride=1,
        cstride=1,
        alpha=alpha_points,
        cmap=colormap,
        edgecolor="k",
    )
    ax2.scatter(
        X_vis[:, 0],
        X_vis[:, 1],
        y,
        c=y,
        cmap=colormap,
        s=point_size,
        edgecolors="k",
        alpha=alpha_points,
    )
    ax2.set_xlabel("X1")
    ax2.set_ylabel("X2")
    ax2.set_zlabel("y / pred")
    ax2.set_title("Regression surface", fontsize=10)
    ax2.view_init(elev=elevation, azim=azimuth)

    ax3 = fig.add_subplot(gs[1, 2])
    ax3.scatter(y_test, y_pred, alpha=0.7, s=20, edgecolors="k")
    min_v = min(y_test.min(), y_pred.min())
    max_v = max(y_test.max(), y_pred.max())
    ax3.plot([min_v, max_v], [min_v, max_v], "r--", lw=2)
    ax3.set_xlabel("True y")
    ax3.set_ylabel("Predicted y")
    ax3.set_title(f"MSE={mse:.3f} | R²={r2:.3f}", fontsize=10)

    ax4 = fig.add_subplot(gs[2, 0])
    residuals = y_pred - y_test
    ax4.hist(residuals, bins=20, alpha=0.8)
    ax4.set_title("Prediction error histogram", fontsize=10)
    ax4.set_xlabel("y_pred - y_true")
    ax4.set_ylabel("Count")

    ax5 = fig.add_subplot(gs[2, 1])
    plot_learning_curve_ax(
        ax5,
        pipeline,
        X_vis,
        y,
        task_type="reg",
        random_state=random_state,
    )

    ax6 = fig.add_subplot(gs[2, 2])

    if family == "linear" and hasattr(base_model, "coef_"):
        coefs = base_model.coef_.ravel()
        ax6.bar(range(len(coefs)), coefs)
        ax6.set_title("Coefficients", fontsize=10)
        ax6.set_xlabel("Feature index")
        ax6.set_ylabel("Weight")
    elif family == "tree_ensemble" and hasattr(base_model, "feature_importances_"):
        imps = base_model.feature_importances_
        ax6.bar(range(len(imps)), imps)
        ax6.set_title("Feature importances", fontsize=10)
        ax6.set_xlabel("Feature index")
        ax6.set_ylabel("Importance")
    elif family == "knn":
        base_k = base_model.n_neighbors
        ks = list(range(max(1, base_k - 4), base_k + 5))
        errors = []
        for k in ks:
            knn_tmp = KNeighborsRegressor(n_neighbors=k, weights=base_model.weights)
            pipe_tmp = make_pipeline(StandardScaler(), knn_tmp)
            pipe_tmp.fit(X_train, y_train)
            y_val_pred = pipe_tmp.predict(X_test)
            err = mean_squared_error(y_test, y_val_pred)
            errors.append(err)
        ax6.plot(ks, errors, "o-")
        ax6.set_title("MSE vs K", fontsize=10)
        ax6.set_xlabel("K")
        ax6.set_ylabel("MSE")
    elif family == "mlp" and hasattr(base_model, "loss_curve_"):
        losses = base_model.loss_curve_
        ax6.plot(range(1, len(losses) + 1), losses, "o-")
        ax6.set_title("MLP loss curve", fontsize=10)
        ax6.set_xlabel("Epoch")
        ax6.set_ylabel("Loss")
    else:
        ax6.text(
            0.5,
            0.5,
            "No extra diagnostics",
            ha="center",
            va="center",
            fontsize=9,
        )
        ax6.set_axis_off()

    plt.tight_layout()

    metrics_md = "### Metrics (regression)\n" + "\n".join(
      [f"- {line}" for line in summary_lines[1:]]
    )

    # Always save PNG of the full visualization
    tmpdir_png = tempfile.mkdtemp()
    png_path = os.path.join(tmpdir_png, "regression_visualization.png")
    fig.savefig(png_path, dpi=140, bbox_inches="tight")

    gif_html = ""
    gif_path = None

    if animate:
        gif_html, gif_path = make_3d_rotation_html_and_gif(
          xx,
          yy,
          z_grid,
          X_vis,
          y,
          colormap,
          elevation,
          anim_frames,
          anim_interval,
          title="3D regression surface",
        )

    return fig, metrics_md, gif_html, png_path, gif_path

# ------------------------------------------------------------------
# Gradio UI
# ------------------------------------------------------------------

theme = gr.themes.Soft(
    primary_hue="cyan",
    secondary_hue="blue",
    neutral_hue="slate",
).set(
    body_background_fill="#020617",
    body_text_color="#e5e7eb",
    block_background_fill="#020617",
    block_border_width="1px",
    block_label_text_size="sm",
    button_primary_background_fill="#06b6d4",
    button_primary_text_color="#ffffff",
    button_primary_background_fill_hover="#0891b2",
)

css = """
.gradio-container {
    max-width: 1700px !important;
    margin: 0 auto !important;
    padding-bottom: 40px;
}

/* Title bar */
#title-bar-wrapper {
    display: flex;
    justify-content: center;
    margin-bottom: 20px;
}
#title-bar {
    background: radial-gradient(circle at top left, #0ea5e9, #0f172a 55%);
    color: white;
    padding: 18px 40px;
    border-radius: 20px;
    box-shadow: 0 18px 60px rgba(8,145,178,0.7);
    text-align: center;
    max-width: 900px;
}

/* Section cards */
.section-card {
    background: #020617;
    border-radius: 16px;
    border: 1px solid #1f2937;
    padding: 16px;
}
/* Shrink Upload CSV box height (but keep text visible) */
#csv-upload {
    height: 140px;          /* جرّب 130–140px، أصغر من الأصلي لكن مريح للعين */
    border-radius: 12px;
    /* لا نستخدم overflow:hidden عشان ما يقصّ النص */
    overflow: visible;
}

/* Visualization cards */
.viz-main-card, .gif-card, .metrics-card {
    background: #020617;
    border-radius: 16px;
    border: 1px solid #1f2937;
    padding: 12px 12px 18px 12px;
    margin-top: 10px;
}

/* Plot and GIF labels */
.viz-main-card .tabitem, .gif-card .tabitem {
    margin-top: 0;
}

/* Make the right column a bit wider */
@media (min-width: 1200px) {
    .data-col { max-width: 22%; }
    .model-col { max-width: 28%; }
    .viz-col  { max-width: 50%; }
}
"""

with gr.Blocks(theme=theme, css=css) as demo:
    with gr.Row(elem_id="title-bar-wrapper"):
        gr.Markdown(
            """
<div id="title-bar">
  <h1 style="margin: 0; font-size: 1.9rem;">
    ML Decision Surfaces Lab 🎯
  </h1>
  <p style="margin: 6px 0 0; opacity: 0.95;">
    Interactive playground for linear, nonlinear & regression models ✨
  </p>
</div>
            """,
            elem_id="title-bar",
        )

    with gr.Row():
        # ---------------- Data settings (left) ----------------
        with gr.Column(scale=1.0, elem_classes=["section-card", "data-col"]):
            gr.Markdown("### Data settings")

            data_source = gr.Radio(
                ["Synthetic data", "Uploaded CSV"],
                value="Synthetic data",
                label="Data source",
            )

            use_pca_flag = gr.Checkbox(
                value=True, label="Use PCA (2D projection)"
            )

            csv_file = gr.File(
              label="Upload CSV",
              file_types=[".csv"],
              file_count="single",
              elem_id="csv-upload",   # مهم عشان نقدر نتحكم فيه من الـ CSS
            )
            csv_info = gr.Markdown(label="CSV info")
            target_col = gr.Dropdown(
                choices=[],
                label="Target column (numeric)",
            )
            feature_col = gr.Dropdown(
                choices=[],
                label="Feature column (numeric)",
            )
            df_state = gr.State(None)

            # Shared data-related sliders (for all modes)
            gr.Markdown("#### Data controls")
            n_samples_global = gr.Slider(
                100, 2000, value=300, step=50, label="Samples"
            )
            noise_global = gr.Slider(
                0, 100, value=10, step=1, label="Noise %"
            )
            outliers_global = gr.Slider(
                0, 100, value=0, step=1, label="Outliers %"
            )
            test_size_global = gr.Slider(
                0.1, 0.5, value=0.3, step=0.05, label="Test size"
            )
            random_state_global = gr.Slider(
                0, 9999, value=42, step=1, label="Random seed"
            )

            def inspect_csv(file):
                if file is None:
                    return (
                        "No CSV uploaded yet.",
                        gr.update(choices=[], value=None),
                        gr.update(choices=[], value=None),
                        None,
                    )
                df = pd.read_csv(file.name)
                numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
                if not numeric_cols:
                    txt = f"Rows: {df.shape[0]}, Cols: {df.shape[1]}\nNo numeric columns found."
                    return txt, gr.update(choices=[], value=None), gr.update(
                        choices=[], value=None
                    ), df

                max_cols_display = 6
                shown = numeric_cols[:max_cols_display]
                more = len(numeric_cols) - max_cols_display
                cols_str = ", ".join(shown)
                if more > 0:
                    cols_str += f" ... (+{more} more)"

                head_md = df.head(3).to_markdown()

                txt = (
                    f"Rows: {df.shape[0]}, Cols: {df.shape[1]}\n"
                    f"Numeric cols: {cols_str}\n\n"
                    f"Preview (head 3):\n{head_md}"
                )

                target_default = numeric_cols[-1]
                feature_choices = numeric_cols[:-1] or numeric_cols
                feature_default = feature_choices[0]

                return (
                    txt,
                    gr.update(choices=numeric_cols, value=target_default),
                    gr.update(choices=feature_choices, value=feature_default),
                    df,
                )

            csv_file.change(
                inspect_csv,
                inputs=csv_file,
                outputs=[csv_info, target_col, feature_col, df_state],
            )

        # ---------------- Model configuration (middle) ----------------
        with gr.Column(scale=1.4, elem_classes=["section-card", "model-col"]):
            gr.Markdown("### Model configuration")

            with gr.Tabs() as mode_tabs:
                # ----- Linear models tab -----
                with gr.Tab("Linear", id="linear"):
                    gr.Markdown("**Linear models**")
                    lin_model = gr.Dropdown(
                        choices=linear_model_names,
                        value="Linear SVM",
                        label="Model",
                    )
                    lin_dataset = gr.Dropdown(
                        choices=["blobs", "shifted", "vertical"],
                        value="blobs",
                        label="Dataset",
                    )
                    lin_boundary = gr.Radio(
                        choices=["Model", "Manual"],
                        value="Model",
                        label="Decision boundary mode",
                    )
                    w1 = gr.Slider(
                        -5, 5, value=1.0, step=0.1, label="w1 (manual)", visible=False
                    )
                    w2 = gr.Slider(
                        -5, 5, value=1.0, step=0.1, label="w2 (manual)", visible=False
                    )
                    b = gr.Slider(
                        -5, 5, value=0.0, step=0.1, label="b (manual)", visible=False
                    )
                    C_lin = gr.Slider(
                        0.01, 10.0, value=1.0, step=0.01, label="C (regularization)"
                    )
                    cmap_lin = gr.Dropdown(
                        ["coolwarm", "bwr", "viridis", "plasma", "cividis"],
                        value="coolwarm",
                        label="Colormap",
                    )

                # ----- Nonlinear models tab -----
                with gr.Tab("Nonlinear", id="nonlinear"):
                    gr.Markdown("**Nonlinear models**")
                    nl_model = gr.Dropdown(
                        choices=[
                            "SVM (RBF)",
                            "SVM (Poly)",
                            "Linear SVM",
                            "KNN",
                            "Decision Tree",
                            "Random Forest",
                            "AdaBoost",
                            "MLP Neural Network",
                            "Naive Bayes",
                            "LDA",
                            "QDA",
                            "Perceptron",
                            "Logistic Regression",
                            "SGD (logistic)",
                            "Gradient Boosting",
                            "Extra Trees",
                        ],
                        value="SVM (RBF)",
                        label="Model",
                    )
                    nl_dataset = gr.Dropdown(
                        choices=["moons", "circles", "blobs", "xor", "spiral", "checkerboard"],
                        value="moons",
                        label="Dataset",
                    )
                    poly_degree_nl = gr.Slider(
                        1, 20, value=3, step=1, label="Polynomial degree (feature map)"
                    )
                    cmap_nl = gr.Dropdown(
                        ["coolwarm", "bwr", "viridis", "plasma", "cividis"],
                        value="coolwarm",
                        label="Colormap",
                    )

                    C_nl = gr.Slider(
                        0.01, 10.0, value=1.0, step=0.01,
                        label="C (SVM/Logistic/SGD)",
                        visible=True,
                    )
                    gamma_nl = gr.Slider(
                        0.01, 5.0, value=1.0, step=0.01,
                        label="Gamma (SVM)",
                        visible=True,
                    )
                    svm_degree = gr.Slider(
                        2, 10, value=3, step=1,
                        label="SVM degree (poly)",
                        visible=False,
                    )
                    n_neighbors = gr.Slider(
                        1, 20, value=7, step=1,
                        label="KNN neighbors",
                        visible=False,
                    )
                    knn_weights = gr.Radio(
                        ["uniform", "distance"],
                        value="uniform",
                        label="KNN weights",
                        visible=False,
                    )
                    max_depth = gr.Slider(
                        1, 20, value=6, step=1,
                        label="Max depth (trees)",
                        visible=False,
                    )
                    min_split = gr.Slider(
                        2, 20, value=2, step=1,
                        label="Min samples split",
                        visible=False,
                    )
                    n_estimators_nl = gr.Slider(
                        10, 500, value=100, step=10,
                        label="N estimators (ensembles)",
                        visible=False,
                    )
                    hidden_units_nl = gr.Slider(
                        5, 200, value=50, step=1,
                        label="MLP hidden units",
                        visible=False,
                    )
                    mlp_alpha_nl = gr.Slider(
                        1e-5, 0.1, value=0.0001, step=1e-4,
                        label="MLP alpha",
                        visible=False,
                    )
                    lr_nl = gr.Slider(
                        1e-4, 0.1, value=0.001, step=1e-4,
                        label="MLP learning rate",
                        visible=False,
                    )

                # ----- Regression tab -----
                with gr.Tab("Regression", id="regression"):
                    gr.Markdown("**Regression models**")
                    reg_model = gr.Dropdown(
                        choices=[
                            "Linear Regression",
                            "Random Forest Regressor",
                            "Decision Tree Regressor",
                            "KNN Regressor",
                            "MLP Regressor",
                            "Ridge Regression",
                            "Lasso Regression",
                            "ElasticNet Regression",
                            "SVR (RBF)",
                        ],
                        value="Linear Regression",
                        label="Model",
                    )
                    reg_dataset = gr.Dropdown(
                        choices=["linear", "quadratic", "interaction"],
                        value="linear",
                        label="Dataset",
                    )
                    cmap_reg = gr.Dropdown(
                        ["viridis", "plasma", "cividis", "coolwarm"],
                        value="viridis",
                        label="Colormap",
                    )
                    alpha_reg_param = gr.Slider(
                        0.0001,
                        1.0,
                        value=0.01,
                        step=0.0001,
                        label="Alpha/L2 (Ridge/Lasso/ElasticNet)",
                        visible=False,
                    )
                    n_estimators_reg = gr.Slider(
                        10, 500, value=100, step=10,
                        label="N estimators (RF)",
                        visible=False,
                    )
                    hidden_units_reg = gr.Slider(
                        5, 200, value=50, step=1,
                        label="MLP hidden units",
                        visible=False,
                    )
                    mlp_alpha_reg = gr.Slider(
                        1e-5, 0.1, value=0.0001, step=1e-4,
                        label="MLP alpha",
                        visible=False,
                    )
                    lr_reg = gr.Slider(
                        1e-4, 0.1, value=0.001, step=1e-4,
                        label="MLP learning rate",
                        visible=False,
                    )

            # --- conditional visibility for some params ---
            def toggle_linear_manual(mode):
                visible = mode == "Manual"
                return (
                    gr.update(visible=visible),
                    gr.update(visible=visible),
                    gr.update(visible=visible),
                )

            lin_boundary.change(
                toggle_linear_manual,
                inputs=lin_boundary,
                outputs=[w1, w2, b],
            )

            def update_nl_visibility(model_name):
                svm_models = ["SVM (RBF)", "SVM (Poly)", "Linear SVM"]
                tree_models = ["Decision Tree", "Random Forest", "Extra Trees"]
                ensemble_models = ["Random Forest", "AdaBoost", "Gradient Boosting", "Extra Trees"]
                mlp_models = ["MLP Neural Network"]
                knn_model = "KNN"

                return (
                    gr.update(visible=model_name in svm_models or model_name in ["Logistic Regression", "SGD (logistic)"]),  # C_nl
                    gr.update(visible=model_name in ["SVM (RBF)", "SVM (Poly)"]),  # gamma
                    gr.update(visible=model_name == "SVM (Poly)"),                # degree
                    gr.update(visible=model_name == knn_model),                   # n_neighbors
                    gr.update(visible=model_name == knn_model),                   # knn_weights
                    gr.update(visible=model_name in tree_models),                 # max_depth
                    gr.update(visible=model_name in tree_models),                 # min_split
                    gr.update(visible=model_name in ensemble_models),             # n_estimators
                    gr.update(visible=model_name in mlp_models),                  # hidden_units
                    gr.update(visible=model_name in mlp_models),                  # mlp_alpha
                    gr.update(visible=model_name in mlp_models),                  # lr
                )

            nl_model.change(
                update_nl_visibility,
                inputs=nl_model,
                outputs=[
                    C_nl,
                    gamma_nl,
                    svm_degree,
                    n_neighbors,
                    knn_weights,
                    max_depth,
                    min_split,
                    n_estimators_nl,
                    hidden_units_nl,
                    mlp_alpha_nl,
                    lr_nl,
                ],
            )

            def update_reg_visibility(model_name):
                ridge_like = ["Ridge Regression", "Lasso Regression", "ElasticNet Regression"]
                rf = "Random Forest Regressor"
                mlp = "MLP Regressor"
                return (
                    gr.update(visible=model_name in ridge_like),  # alpha
                    gr.update(visible=model_name == rf),          # n_estimators
                    gr.update(visible=model_name == mlp),         # hidden_units
                    gr.update(visible=model_name == mlp),         # mlp_alpha
                    gr.update(visible=model_name == mlp),         # lr
                )

            reg_model.change(
                update_reg_visibility,
                inputs=reg_model,
                outputs=[
                    alpha_reg_param,
                    n_estimators_reg,
                    hidden_units_reg,
                    mlp_alpha_reg,
                    lr_reg,
                ],
            )

        # ---------------- Visualization (right) ----------------
        with gr.Column(scale=2.0, elem_classes=["section-card", "viz-col"]):
            gr.Markdown("### Visualization")

            current_mode = gr.State("linear")

            run_button = gr.Button(
                "Run experiment", variant="primary"
            )

            with gr.Row():
                animate_flag = gr.Checkbox(
                    value=True, label="Generate 3D rotation animation"
                )
                anim_interval = gr.Slider(
                    50, 1000, value=200, step=50, label="Animation interval (ms)"
                )
                anim_frames = gr.Slider(
                    12, 60, value=24, step=4, label="Animation frames"
                )

            with gr.Row():
                azimuth_global = gr.Slider(
                    0, 360, value=45, step=1, label="Azimuth"
                )
                elevation_global = gr.Slider(
                    0, 90, value=30, step=1, label="Elevation"
                )

            with gr.Row():
                alpha_global = gr.Slider(
                    0.1, 1.0, value=0.6, step=0.05, label="Alpha"
                )
                point_size_global = gr.Slider(
                    5, 100, value=20, step=1, label="Point size"
                )

            with gr.Group(elem_classes=["viz-main-card"]):
                  main_plot = gr.Plot(label="", show_label=False)


            with gr.Group(elem_classes=["gif-card"]):
                gif_html_box = gr.HTML(label="3D rotation GIF (3D surface only)")

            with gr.Group(elem_classes=["metrics-card"]):
                metrics_box = gr.Markdown(label="Metrics & summary")

            with gr.Row():
                download_png_btn = gr.DownloadButton(
                    "Download current plot (PNG)"
                )
                download_gif_btn = gr.DownloadButton(
                    "Download 3D GIF"
                )

            # ----- change button text when switching tabs -----
            def on_tab_change(selected):
                if selected in ["linear", "Linear"]:
                    label = "Run linear experiment"
                    mode = "linear"
                elif selected in ["nonlinear", "Nonlinear"]:
                    label = "Run nonlinear experiment"
                    mode = "nonlinear"
                else:
                    label = "Run regression experiment"
                    mode = "regression"
                return label, mode

            mode_tabs.select(
                on_tab_change,
                inputs=None,
                outputs=[run_button, current_mode],
            )

            # ----- main callback -----
            def run_experiment(
                mode,
                data_source,
                use_pca_flag,
                csv_file,
                target_col,
                feature_col,
                df_state,
                n_samples_global,
                noise_global,
                outliers_global,
                test_size_global,
                random_state_global,
                lin_model,
                lin_dataset,
                lin_boundary,
                w1,
                w2,
                b,
                C_lin,
                cmap_lin,
                nl_model,
                nl_dataset,
                poly_degree_nl,
                cmap_nl,
                C_nl,
                gamma_nl,
                svm_degree,
                n_neighbors,
                knn_weights,
                max_depth,
                min_split,
                n_estimators_nl,
                hidden_units_nl,
                mlp_alpha_nl,
                lr_nl,
                reg_model,
                reg_dataset,
                cmap_reg,
                alpha_reg_param,
                n_estimators_reg,
                hidden_units_reg,
                mlp_alpha_reg,
                lr_reg,
                azimuth_global,
                elevation_global,
                alpha_global,
                point_size_global,
                animate_flag,
                anim_interval,
                anim_frames,
            ):
                df = df_state
                if csv_file is not None and df is None and data_source == "Uploaded CSV":
                    df = pd.read_csv(csv_file.name)

                if mode == "linear":
                    fig, metrics_md, gif_html, png_path, gif_path = linear_experiment_core(
                        data_source=data_source,
                        df=df,
                        use_pca=use_pca_flag,
                        target_col=target_col or None,
                        feature_col=feature_col or None,
                        model_name=lin_model,
                        dataset=lin_dataset,
                        noise_pct=noise_global,
                        outliers_pct=outliers_global,
                        n_samples=int(n_samples_global),
                        test_size=test_size_global,
                        random_state=int(random_state_global),
                        azimuth=int(azimuth_global),
                        elevation=int(elevation_global),
                        alpha_points=alpha_global,
                        point_size=int(point_size_global),
                        C=C_lin,
                        colormap=cmap_lin,
                        boundary_mode=lin_boundary,
                        w1_manual=w1,
                        w2_manual=w2,
                        b_manual=b,
                        animate=animate_flag,
                        anim_interval=int(anim_interval),
                        anim_frames=int(anim_frames),
                    )
                elif mode == "nonlinear":
                    fig, metrics_md, gif_html, png_path, gif_path = nonlinear_experiment_core(
                        data_source=data_source,
                        df=df,
                        use_pca=use_pca_flag,
                        target_col=target_col or None,
                        feature_col=feature_col or None,
                        model_name=nl_model,
                        dataset=nl_dataset,
                        poly_degree=int(poly_degree_nl),
                        noise_pct=noise_global,
                        outliers_pct=outliers_global,
                        n_samples=int(n_samples_global),
                        test_size=test_size_global,
                        random_state=int(random_state_global),
                        azimuth=int(azimuth_global),
                        elevation=int(elevation_global),
                        alpha_points=alpha_global,
                        point_size=int(point_size_global),
                        colormap=cmap_nl,
                        C=C_nl,
                        gamma=gamma_nl,
                        svm_degree=int(svm_degree),
                        n_neighbors=int(n_neighbors),
                        weights=knn_weights,
                        max_depth=int(max_depth) if max_depth is not None else None,
                        min_samples_split=int(min_split) if min_split is not None else 2,
                        n_estimators=int(n_estimators_nl) if n_estimators_nl is not None else 100,
                        hidden_layer_sizes=int(hidden_units_nl) if hidden_units_nl is not None else 50,
                        mlp_alpha=mlp_alpha_nl if mlp_alpha_nl is not None else 0.0001,
                        learning_rate_init=lr_nl if lr_nl is not None else 0.001,
                        animate=animate_flag,
                        anim_interval=int(anim_interval),
                        anim_frames=int(anim_frames),
                    )
                else:
                    fig, metrics_md, gif_html, png_path, gif_path = regression_experiment_core(
                        data_source=data_source,
                        df=df,
                        use_pca=use_pca_flag,
                        target_col=target_col or None,
                        feature_col=feature_col or None,
                        model_name=reg_model,
                        dataset=reg_dataset,
                        noise_pct=noise_global,
                        outliers_pct=outliers_global,
                        n_samples=int(n_samples_global),
                        test_size=test_size_global,
                        random_state=int(random_state_global),
                        azimuth=int(azimuth_global),
                        elevation=int(elevation_global),
                        alpha_points=alpha_global,
                        point_size=int(point_size_global),
                        colormap=cmap_reg,
                        alpha_reg=alpha_reg_param if alpha_reg_param is not None else 0.01,
                        n_estimators=int(n_estimators_reg) if n_estimators_reg is not None else 100,
                        hidden_units=int(hidden_units_reg) if hidden_units_reg is not None else 50,
                        mlp_alpha=mlp_alpha_reg if mlp_alpha_reg is not None else 0.0001,
                        learning_rate_init=lr_reg if lr_reg is not None else 0.001,
                        animate=animate_flag,
                        anim_interval=int(anim_interval),
                        anim_frames=int(anim_frames),
                    )

                return (
                    fig,
                    (gif_html or ""),
                    metrics_md,
                    (png_path or None),
                    (gif_path or None),
                )

            run_button.click(
                run_experiment,
                inputs=[
                    current_mode,
                    data_source,
                    use_pca_flag,
                    csv_file,
                    target_col,
                    feature_col,
                    df_state,
                    n_samples_global,
                    noise_global,
                    outliers_global,
                    test_size_global,
                    random_state_global,
                    lin_model,
                    lin_dataset,
                    lin_boundary,
                    w1,
                    w2,
                    b,
                    C_lin,
                    cmap_lin,
                    nl_model,
                    nl_dataset,
                    poly_degree_nl,
                    cmap_nl,
                    C_nl,
                    gamma_nl,
                    svm_degree,
                    n_neighbors,
                    knn_weights,
                    max_depth,
                    min_split,
                    n_estimators_nl,
                    hidden_units_nl,
                    mlp_alpha_nl,
                    lr_nl,
                    reg_model,
                    reg_dataset,
                    cmap_reg,
                    alpha_reg_param,
                    n_estimators_reg,
                    hidden_units_reg,
                    mlp_alpha_reg,
                    lr_reg,
                    azimuth_global,
                    elevation_global,
                    alpha_global,
                    point_size_global,
                    animate_flag,
                    anim_interval,
                    anim_frames,
                ],
                outputs=[
                    main_plot,
                    gif_html_box,
                    metrics_box,
                    download_png_btn,
                    download_gif_btn,
                ],
            )

In [ ]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://935b6d332070b7f126.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
